# V11 bilingual notice checker

V11 compares the information conveyed in both directions. Its frozen equivalence classifier uses contradiction, entailment, neutral, similarity and explicit-fact signals. A corroborated qualifier check helps identify missing conditions. No V7 fallback is called.

The score is a decision score, not a calibrated probability. A clean verdict means no discrepancy was found. The supplied report describes the fresh benchmark and its limitations.

The included source and pretrained model revisions are frozen. This notebook is a Colab-ready copy; the reported benchmark was executed in the local CPU runtime, not Colab. The NLLB component is a research model licensed CC-BY-NC-4.0. See https://huggingface.co/facebook/nllb-200-distilled-600M.


In [ ]:
!pip -q install sentence-transformers==5.7.0 transformers==5.22.1 sentencepiece==0.2.3 opencc-python-reimplemented==0.1.7 pandas scipy numpy


In [ ]:
from pathlib import Path
import base64, json, sys, hashlib
folder=Path('/content/v11_checker')
folder.mkdir(exist_ok=True)
payload={'__init__.py': 'IiIiVjExIGJpbGluZ3VhbCBub3RpY2UgY2hlY2tlci4iIiIK', 'engine.py': 'IiIiRnJvemVuIFYxMTogZXF1aXZhbGVuY2UgY2xhc3NpZmljYXRpb24gcGx1cyBjb3Jyb2JvcmF0ZWQgbWlzc2luZy1kZXRhaWwgY2hlY2tzLgoKVGhlIHNjb3JlIGlzIGEgbGVhcm5lZCBkZWNpc2lvbiBzY29yZSwgbm90IGEgY2FsaWJyYXRlZCBwcm9iYWJpbGl0eSBvZiBlcnJvci4KTm8gYmVuY2htYXJrIGxhYmVscywgc2NlbmFyaW8gbmFtZXMgb3IgY2FzZSBJRHMgZW50ZXIgdGhlIHNjb3JpbmcgZmVhdHVyZXMuCiIiIgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IGpzb24KaW1wb3J0IHJlCmltcG9ydCBkaWZmbGliCmltcG9ydCBudW1weSBhcyBucAp0cnk6CiAgICBmcm9tIC5iYWNrYm9uZSBpbXBvcnQgU2lnbmFsRXh0cmFjdG9yLCBNT0RFTF9JRFMsIE1PREVMX1JFVklTSU9OUwogICAgZnJvbSAuZmVhdHVyZXMgaW1wb3J0IHZlY3RvciwgZmVhdHVyZV9uYW1lcwogICAgZnJvbSAuY292ZXJhZ2VfcnVsZXMgaW1wb3J0IHVuc3VwcG9ydGVkX2RldGFpbHMKZXhjZXB0IEltcG9ydEVycm9yOgogICAgZnJvbSBiYWNrYm9uZSBpbXBvcnQgU2lnbmFsRXh0cmFjdG9yLCBNT0RFTF9JRFMsIE1PREVMX1JFVklTSU9OUwogICAgZnJvbSBmZWF0dXJlcyBpbXBvcnQgdmVjdG9yLCBmZWF0dXJlX25hbWVzCiAgICBmcm9tIGNvdmVyYWdlX3J1bGVzIGltcG9ydCB1bnN1cHBvcnRlZF9kZXRhaWxzCgoKY2xhc3MgRGVjaXNpb246CiAgICBkZWYgX19pbml0X18oc2VsZik6CiAgICAgICAgc2VsZi5oZWFkID0ganNvbi5sb2FkcyhQYXRoKF9fZmlsZV9fKS53aXRoX25hbWUoJ2xlYXJuZWRfaGVhZC5qc29uJykucmVhZF90ZXh0KCkpCiAgICAgICAgYXNzZXJ0IHNlbGYuaGVhZFsnZmVhdHVyZV9uYW1lcyddID09IGZlYXR1cmVfbmFtZXMoKQoKICAgIGRlZiBzY29yZShzZWxmLCByZXN1bHQpOgogICAgICAgIGggPSBzZWxmLmhlYWQKICAgICAgICB4ID0gKHZlY3RvcihyZXN1bHQpIC0gbnAuYXJyYXkoaFsnbWVhbiddKSkgLyBucC5hcnJheShoWydzY2FsZSddKQogICAgICAgIHZhbHVlID0gZmxvYXQoeCBAIG5wLmFycmF5KGhbJ2NvZWZmaWNpZW50cyddKSArIGhbJ2ludGVyY2VwdCddKQogICAgICAgIHJldHVybiBmbG9hdCgxIC8gKDEgKyBucC5leHAoLW5wLmNsaXAodmFsdWUsIC01MCwgNTApKSkpCgogICAgZGVmIGFwcGx5KHNlbGYsIHJlc3VsdCwgZW5nbGlzaCwgY2hpbmVzZSk6CiAgICAgICAgaWYgcmVzdWx0Wydjb3ZlcmFnZSddICE9ICdjb21wbGV0ZV9ub3RpY2UnOgogICAgICAgICAgICByZXR1cm4gcmVzdWx0CiAgICAgICAgcyA9IHJlc3VsdFsnc2lnbmFscyddCiAgICAgICAgc2NvcmUgPSBzZWxmLnNjb3JlKHJlc3VsdCkKICAgICAgICBkZXRhaWxfd2FybmluZ3MgPSB1bnN1cHBvcnRlZF9kZXRhaWxzKHMsIGVuZ2xpc2gsIGNoaW5lc2UpCiAgICAgICAgZm9yIHBhc3NhZ2UgaW4gc1sncGFzc2FnZXMnXToKICAgICAgICAgICAgZGV0YWlsX3dhcm5pbmdzICs9IHVuc3VwcG9ydGVkX2RldGFpbHMocGFzc2FnZSwgcGFzc2FnZVsnZW5nbGlzaF9zcGFuJ10sIHBhc3NhZ2VbJ2NoaW5lc2Vfc3BhbiddKQogICAgICAgICMgS2VlcCBhIGRldGFpbCBvbmNlIHdoZW4gYm90aCBmdWxsLW5vdGljZSBhbmQgYWxpZ25lZCB2aWV3cyBzdXBwb3J0IGl0LgogICAgICAgIGRldGFpbF93YXJuaW5ncyA9IGxpc3Qoeyh3WydkZXRhaWxfbGFuZ3VhZ2UnXSwgd1sndW5zdXBwb3J0ZWRfZGV0YWlsJ10pOiB3IGZvciB3IGluIGRldGFpbF93YXJuaW5nc30udmFsdWVzKCkpCiAgICAgICAgZmxhZ2dlZCA9IHNjb3JlID49IHNlbGYuaGVhZFsndGhyZXNob2xkJ10gb3IgYm9vbChkZXRhaWxfd2FybmluZ3MpCiAgICAgICAgd2FybmluZ3MgPSBbXQogICAgICAgIGlmIGZsYWdnZWQ6CiAgICAgICAgICAgIHdhcm5pbmdzID0gW3cgZm9yIHcgaW4gcmVzdWx0Wyd3YXJuaW5ncyddIGlmIHdbJ2tpbmQnXSA9PSAnZXhwbGljaXRfZmFjdCddICsgZGV0YWlsX3dhcm5pbmdzCiAgICAgICAgICAgIGlmIG5vdCB3YXJuaW5nczoKICAgICAgICAgICAgICAgIHdhcm5pbmdzID0gW3NlbGYuZXhwbGFpbihyZXN1bHQsIGVuZ2xpc2gsIGNoaW5lc2UpXQogICAgICAgIHJldHVybiBkaWN0KHJlc3VsdCwgZmxhZ2dlZD1mbGFnZ2VkLCB3YXJuaW5ncz13YXJuaW5ncywgZXJyb3Jfc2NvcmU9c2NvcmUsCiAgICAgICAgICAgIGRlY2lzaW9uX3RocmVzaG9sZD1zZWxmLmhlYWRbJ3RocmVzaG9sZCddLAogICAgICAgICAgICBkZWNpc2lvbl9wYXRoPSd2MTFfdW5zdXBwb3J0ZWRfZGV0YWlsJyBpZiBkZXRhaWxfd2FybmluZ3MgZWxzZSAndjExX2xlYXJuZWRfZXF1aXZhbGVuY2UnLAogICAgICAgICAgICB2N19mYWxsYmFjaz1GYWxzZSkKCiAgICBkZWYgZXhwbGFpbihzZWxmLCByZXN1bHQsIGVuZ2xpc2gsIGNoaW5lc2UpOgogICAgICAgIHMgPSByZXN1bHRbJ3NpZ25hbHMnXQogICAgICAgIG9wdGlvbnMgPSBbXQogICAgICAgIGZvciBwIGluIHNbJ3Bhc3NhZ2VzJ106CiAgICAgICAgICAgIHBzID0gZGljdChwLCBwYXNzYWdlcz1bcF0pCiAgICAgICAgICAgIG9wdGlvbnMuYXBwZW5kKChzZWxmLnNjb3JlKGRpY3Qoc2lnbmFscz1wcywgd2FybmluZ3M9W10pKSwgcCkpCiAgICAgICAgIyBBIHBhc3NhZ2UgaXMgYSBuYXZpZ2F0aW9uIGFpZDsgdGhlIGFjdHVhbCBkZWNpc2lvbiB1c2VzIHRoZSBmdWxsIGlucHV0LgogICAgICAgIGZvY3VzID0gbWF4KG9wdGlvbnMsIGtleT1sYW1iZGEgeDogeFswXSlbMV0gaWYgb3B0aW9ucyBlbHNlIGRpY3QocywgZW5nbGlzaF9zcGFuPWVuZ2xpc2gsIGNoaW5lc2Vfc3Bhbj1jaGluZXNlKQogICAgICAgIGUsIHosIG10ID0gZm9jdXNbJ2VuZ2xpc2hfc3BhbiddLCBmb2N1c1snY2hpbmVzZV9zcGFuJ10sIGZvY3VzWyd0cmFuc2xhdGlvbiddCiAgICAgICAgZWRpdHMgPSBbXQogICAgICAgIGEsIGIgPSByZS5maW5kYWxsKHIiXHcrKD86WyfigJktXVx3KykqfFteXHdcc10iLCBlKSwgcmUuZmluZGFsbChyIlx3Kyg/Olsn4oCZLV1cdyspKnxbXlx3XHNdIiwgbXQpCiAgICAgICAgZm9yIHRhZywgaSwgaiwgaywgbCBpbiBkaWZmbGliLlNlcXVlbmNlTWF0Y2hlcihhPVt4Lmxvd2VyKCkgZm9yIHggaW4gYV0sIGI9W3gubG93ZXIoKSBmb3IgeCBpbiBiXSwgYXV0b2p1bms9RmFsc2UpLmdldF9vcGNvZGVzKCk6CiAgICAgICAgICAgIGlmIHRhZyAhPSAnZXF1YWwnOgogICAgICAgICAgICAgICAgZWRpdHMuYXBwZW5kKGRpY3QoZW5nbGlzaD0nICcuam9pbihhW21heCgwLGktMik6bWluKGxlbihhKSxqKzIpXSksIG1hY2hpbmVfYmFja3RyYW5zbGF0aW9uPScgJy5qb2luKGJbbWF4KDAsay0yKTptaW4obGVuKGIpLGwrMildKSkpCiAgICAgICAgZGlyZWN0aW9uID0gaW50KG5wLmFyZ21heChucC5hcnJheShmb2N1c1snY3Jvc3NfbmV1dHJhbCddKSArIG5wLmFycmF5KGZvY3VzWyduZXV0cmFsJ10pKSkKICAgICAgICBpZiBtaW4oZm9jdXNbJ2Nyb3NzX25ldXRyYWwnXVtkaXJlY3Rpb25dLCBmb2N1c1snbmV1dHJhbCddW2RpcmVjdGlvbl0pID49IC41OgogICAgICAgICAgICBjYXRlZ29yeSA9ICdNaXNzaW5nIG9yIGNoYW5nZWQgZGV0YWlsJwogICAgICAgICAgICBpZiByZS5zZWFyY2gocidcYig/OnVubGVzc3xleGNlcHR8b25seSBpZilcYnzpmaTpnZ586LGB5YWNfOS+i+WklicsIGUrJyAnK3osIHJlLkkpOgogICAgICAgICAgICAgICAgY2F0ZWdvcnkgPSAnRXhjZXB0aW9uIG9yIGNvbmRpdGlvbiBtYXkgZGlmZmVyJwogICAgICAgICAgICBlbGlmIHJlLnNlYXJjaChyJ1xiKD86b25seXxhbGx8ZXZlcnl8Ym90aHxlaXRoZXJ8aW5jbHVkaW5nfGFueSlcYnzlg4V85Y+qfOWdh3zmiYDmnIl85YyF5ousJywgZSsnICcreiwgcmUuSSk6CiAgICAgICAgICAgICAgICBjYXRlZ29yeSA9ICdTY29wZSBvciByZXF1aXJlbWVudCBtYXkgZGlmZmVyJwogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGNhdGVnb3J5ID0gJ0ZhY3RzIG9yIGluc3RydWN0aW9ucyBtYXkgZGlmZmVyJwogICAgICAgICMgQWx3YXlzIGV4cG9zZSBib3RoIGFjdHVhbCBwYXNzYWdlczsgYSBnZW5lcmF0ZWQgZ2xvc3MgY2FuIGJlIHdyb25nLgogICAgICAgIHdoeSA9IGYne2NhdGVnb3J5fS4gRW5nbGlzaCBzdGF0ZXM6IOKAnHtlfeKAnS4gVGhlIENoaW5lc2UgcGFzc2FnZSBoYXMgdGhlIG1hY2hpbmUgZ2xvc3M6IOKAnHttdH3igJ0uIENvbXBhcmUgdGhlIGhpZ2hsaWdodGVkIHdvcmRpbmcgYWdhaW5zdCB0aGUgb3JpZ2luYWwgQ2hpbmVzZTsgdGhlIGdsb3NzIGNhbiBjb250YWluIGVycm9ycy4nCiAgICAgICAgcmV0dXJuIGRpY3Qoa2luZD0nZXF1aXZhbGVuY2VfbW9kZWwnLCBjYXRlZ29yeT1jYXRlZ29yeSwgd2h5PXdoeSwKICAgICAgICAgICAgZW5nbGlzaF9zcGFuPWUsIGNoaW5lc2Vfc3Bhbj16LCBtYWNoaW5lX2JhY2t0cmFuc2xhdGlvbj1tdCwKICAgICAgICAgICAgY29tcGFyaXNvbl9lZGl0cz1lZGl0cykKCgpjbGFzcyBDaGVja2VyKFNpZ25hbEV4dHJhY3Rvcik6CiAgICBkZWYgX19pbml0X18oc2VsZiwgKmFyZ3MsICoqa3dhcmdzKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCphcmdzLCAqKmt3YXJncykKICAgICAgICBzZWxmLmRlY2lzaW9uID0gRGVjaXNpb24oKQoKICAgIGRlZiBwcmVkaWN0X21hbnkoc2VsZiwgaW5wdXRzKToKICAgICAgICByZXN1bHRzID0gc3VwZXIoKS5wcmVkaWN0X21hbnkoaW5wdXRzKQogICAgICAgIHJldHVybiBbc2VsZi5kZWNpc2lvbi5hcHBseShyLCByb3dbJ2VuZ2xpc2gnXSwgcm93WydjaGluZXNlJ10pIGZvciByLCByb3cgaW4gemlwKHJlc3VsdHMsIGlucHV0cyldCgogICAgZGVmIGNoZWNrX25vdGljZShzZWxmLCBjaGluZXNlX3RleHQsIGVuZ2xpc2hfdGV4dCk6CiAgICAgICAgaW1wb3J0IHBhbmRhcyBhcyBwZAogICAgICAgIHIgPSBzZWxmLnByZWRpY3RfbWFueShbZGljdChjYXNlX2lkPSdpbnRlcmFjdGl2ZScsIGNoaW5lc2U9Y2hpbmVzZV90ZXh0LCBlbmdsaXNoPWVuZ2xpc2hfdGV4dCldKVswXQogICAgICAgIHdhcm5pbmdzID0gclsnd2FybmluZ3MnXSBvciBbZGljdCh3aHk9JycsIGNhdGVnb3J5PScnKV0KICAgICAgICByZXR1cm4gcGQuRGF0YUZyYW1lKFtkaWN0KHN0YXR1cz0n4pqg77iPIENoZWNrJyBpZiByWydmbGFnZ2VkJ10gZWxzZSAn4pyFIE5vIGRpc2NyZXBhbmN5IGZvdW5kJywKICAgICAgICAgICAgY2hpbmVzZT1jaGluZXNlX3RleHQsIGVuZ2xpc2g9ZW5nbGlzaF90ZXh0LCBjb3ZlcmFnZT1yWydjb3ZlcmFnZSddLAogICAgICAgICAgICBkZWNpc2lvbl9wYXRoPXJbJ2RlY2lzaW9uX3BhdGgnXSwgKip3KSBmb3IgdyBpbiB3YXJuaW5nc10pCg==', 'backbone.py': 'IiIiVjExOiBkaXJlY3QgY29udGV4dHVhbCBpbmZlcmVuY2UgYW5kIGJpbGluZ3VhbCBmYWN0cywgd2l0aCBubyBWNyBkZXBlbmRlbmN5LgoKVGhlIENoaW5lc2UtdG8tRW5nbGlzaCB0cmFuc2xhdGlvbiBpcyBhIGNyb3NzLWNoZWNrLCBuZXZlciBhIHJlZmVyZW5jZSBhbnN3ZXIuClNjb3JlcyByZW1haW4gZGlhZ25vc3RpYyBtb2RlbCBzY29yZXMsIG5vdCBjYWxpYnJhdGVkIHByb2JhYmlsaXRpZXMgb2YgZXJyb3IuCiIiIgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IHJlCmltcG9ydCBkaWZmbGliCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IEF1dG9Ub2tlbml6ZXIsIEF1dG9Nb2RlbEZvclNlcXVlbmNlQ2xhc3NpZmljYXRpb24sIEF1dG9Nb2RlbEZvclNlcTJTZXFMTQpmcm9tIHNlbnRlbmNlX3RyYW5zZm9ybWVycyBpbXBvcnQgU2VudGVuY2VUcmFuc2Zvcm1lcgpmcm9tIHNjaXB5Lm9wdGltaXplIGltcG9ydCBsaW5lYXJfc3VtX2Fzc2lnbm1lbnQKdHJ5OgogICAgZnJvbSAuZmFjdHMgaW1wb3J0IG5vcm1hbCwgZGlyZWN0X2V2aWRlbmNlCiAgICBmcm9tIC50ZXJtaW5vbG9neSBpbXBvcnQgdHJhbnNsYXRpb25fc291cmNlCmV4Y2VwdCBJbXBvcnRFcnJvcjoKICAgIGZyb20gZmFjdHMgaW1wb3J0IG5vcm1hbCwgZGlyZWN0X2V2aWRlbmNlCiAgICBmcm9tIHRlcm1pbm9sb2d5IGltcG9ydCB0cmFuc2xhdGlvbl9zb3VyY2UKCk1PREVMX1JFVklTSU9OUyA9IGRpY3QoZW1iZWRkaW5nPSc4MzYxMjFhMDUzM2U1NjY0YjIxYzdhYWNjNWQyMjk1MWYyYjhiMjViJyxubGk9J2I1MTEzZWIzOGFiNjNlZmRkN2YyODBmOGMxNDRlYThiMTNmOTc4Y2UnLHRyYW5zbGF0aW9uPSdmOGQzMzNhMDk4ZDE5YjRmZDlhOGIxOGY5NDE3MDQ4N2FkM2Y4MjFkJykKCk1PREVMX0lEUyA9IGRpY3QoZW1iZWRkaW5nPSdzZW50ZW5jZS10cmFuc2Zvcm1lcnMvTGFCU0UnLAogICAgbmxpPSdNb3JpdHpMYXVyZXIvbURlQkVSVGEtdjMtYmFzZS14bmxpLW11bHRpbGluZ3VhbC1ubGktMm1pbDcnLAogICAgdHJhbnNsYXRpb249J2ZhY2Vib29rL25sbGItMjAwLWRpc3RpbGxlZC02MDBNJykKCgpkZWYgc2VudGVuY2VzKHRleHQpOgogICAgIyBLZWVwIHNlbWljb2xvbnMgYW5kIGNvbW1hcyB3aXRoaW4gcHJvcG9zaXRpb25zOyBwcmVzZXJ2ZSByZW9yZGVyaW5ncy4KICAgIHJldHVybiBbeC5zdHJpcCgpIGZvciB4IGluIHJlLnNwbGl0KHInKD88PVvjgILvvIHvvJ9dKXwoPzw9Wy4hP10pXHMrKD89W0EtWl0pfFxuKycsIHRleHQpIGlmIHguc3RyaXAoKV0KCgpkZWYgYWxpZ25lZF9ncm91cHMoenMsIGVzLCBlbWJlZGRpbmdzKToKICAgIGlmIG5vdCB6cyBvciBub3QgZXM6cmV0dXJuIFtdCiAgICBzaW09bnAuYXJyYXkoW1tmbG9hdChlbWJlZGRpbmdzW3pdQGVtYmVkZGluZ3NbZV0pIGZvciBlIGluIGVzXSBmb3IgeiBpbiB6c10pCiAgICBpaSxqaj1saW5lYXJfc3VtX2Fzc2lnbm1lbnQoLXNpbSkKICAgIGdyb3Vwcz1bKFtpbnQoaSldLFtpbnQoaildLGZsb2F0KHNpbVtpLGpdKSkgZm9yIGksaiBpbiB6aXAoaWksamopXQogICAgIyBQcmVzZXJ2ZSB1bm1hdGNoZWQgbWF0ZXJpYWwgYnkgYXR0YWNoaW5nIGl0IHRvIHRoZSBjbG9zZXN0IG1hdGNoZWQgZ3JvdXAuCiAgICAjIFRoZSBmdWxsLW5vdGljZSBzZW1hbnRpYyBwYXNzIGRlY2lkZXMgbWVhbmluZywgbm90IHNlbnRlbmNlLWNvdW50IHBhcml0eS4KICAgIGZvciBpIGluIHNvcnRlZChzZXQocmFuZ2UobGVuKHpzKSkpLXNldChpaSkpOgogICAgICAgIGs9bWF4KHJhbmdlKGxlbihncm91cHMpKSxrZXk9bGFtYmRhIGs6bWF4KHNpbVtpLGpdIGZvciBqIGluIGdyb3Vwc1trXVsxXSkpCiAgICAgICAgZ3JvdXBzW2tdWzBdLmFwcGVuZChpKQogICAgZm9yIGogaW4gc29ydGVkKHNldChyYW5nZShsZW4oZXMpKSktc2V0KGpqKSk6CiAgICAgICAgaz1tYXgocmFuZ2UobGVuKGdyb3VwcykpLGtleT1sYW1iZGEgazptYXgoc2ltW2ksal0gZm9yIGkgaW4gZ3JvdXBzW2tdWzBdKSkKICAgICAgICBncm91cHNba11bMV0uYXBwZW5kKGopCiAgICByZXR1cm4gWygnICcuam9pbih6c1tpXSBmb3IgaSBpbiBzb3J0ZWQoaXopKSwnICcuam9pbihlc1tqXSBmb3IgaiBpbiBzb3J0ZWQoaWUpKSxzY29yZSkgZm9yIGl6LGllLHNjb3JlIGluIGdyb3Vwc10KCgpkZWYgZGlnZXN0KHRleHQpOnJldHVybiBoYXNobGliLnNoYTI1Nih0ZXh0LmVuY29kZSgpKS5oZXhkaWdlc3QoKQoKCmNsYXNzIFNpZ25hbEV4dHJhY3RvcjoKICAgIGRlZiBfX2luaXRfXyhzZWxmLGNhY2hlPU5vbmUsY29uZmlnPU5vbmUsZGV2aWNlPU5vbmUscHJvZ3Jlc3M9cHJpbnQpOgogICAgICAgIHNlbGYucHJvZ3Jlc3M9cHJvZ3Jlc3MKICAgICAgICBzZWxmLmRldmljZT1kZXZpY2Ugb3IgKCdjdWRhJyBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgJ2NwdScpCiAgICAgICAgdG9yY2guc2V0X251bV90aHJlYWRzKDQpCiAgICAgICAgdG9yY2gubWFudWFsX3NlZWQoMTApCiAgICAgICAgc2VsZi5jb25maWc9Y29uZmlnIG9yIGpzb24ubG9hZHMoUGF0aChfX2ZpbGVfXykud2l0aF9uYW1lKCdiYWNrYm9uZV9jb25maWcuanNvbicpLnJlYWRfdGV4dCgpKQogICAgICAgIHNlbGYuY2FjaGVfcGF0aD1QYXRoKGNhY2hlKSBpZiBjYWNoZSBlbHNlIE5vbmUKICAgICAgICBzZWxmLmNhY2hlPWpzb24ubG9hZHMoc2VsZi5jYWNoZV9wYXRoLnJlYWRfdGV4dCgpKSBpZiBzZWxmLmNhY2hlX3BhdGggYW5kIHNlbGYuY2FjaGVfcGF0aC5leGlzdHMoKSBlbHNlIHt9CiAgICAgICAgc2VsZi5wcm9ncmVzcygnTG9hZGluZyBWMTEgbW9kZWxzIChubyBWNyBmYWxsYmFjaykuJyxmbHVzaD1UcnVlKQogICAgICAgIHNlbGYubGFic2U9U2VudGVuY2VUcmFuc2Zvcm1lcihNT0RFTF9JRFNbJ2VtYmVkZGluZyddLHJldmlzaW9uPU1PREVMX1JFVklTSU9OU1snZW1iZWRkaW5nJ10sZGV2aWNlPXNlbGYuZGV2aWNlKQogICAgICAgIHNlbGYudG9rPUF1dG9Ub2tlbml6ZXIuZnJvbV9wcmV0cmFpbmVkKE1PREVMX0lEU1snbmxpJ10scmV2aXNpb249TU9ERUxfUkVWSVNJT05TWydubGknXSkKICAgICAgICBzZWxmLm5saT1BdXRvTW9kZWxGb3JTZXF1ZW5jZUNsYXNzaWZpY2F0aW9uLmZyb21fcHJldHJhaW5lZChNT0RFTF9JRFNbJ25saSddLHJldmlzaW9uPU1PREVMX1JFVklTSU9OU1snbmxpJ10pLnRvKHNlbGYuZGV2aWNlKS5ldmFsKCkKICAgICAgICBzZWxmLm10X3Rvaz1BdXRvVG9rZW5pemVyLmZyb21fcHJldHJhaW5lZChNT0RFTF9JRFNbJ3RyYW5zbGF0aW9uJ10scmV2aXNpb249TU9ERUxfUkVWSVNJT05TWyd0cmFuc2xhdGlvbiddLHNyY19sYW5nPSd6aG9fSGFucycpCiAgICAgICAgc2VsZi5tdD1BdXRvTW9kZWxGb3JTZXEyU2VxTE0uZnJvbV9wcmV0cmFpbmVkKE1PREVMX0lEU1sndHJhbnNsYXRpb24nXSxyZXZpc2lvbj1NT0RFTF9SRVZJU0lPTlNbJ3RyYW5zbGF0aW9uJ10pLnRvKHNlbGYuZGV2aWNlKS5ldmFsKCkKICAgICAgICBsYWJlbHM9c2VsZi5ubGkuY29uZmlnLmlkMmxhYmVsCiAgICAgICAgc2VsZi5sYWJlbHM9e25hbWU6aSBmb3IgaSxuYW1lIGluIGxhYmVscy5pdGVtcygpfQogICAgICAgIGFzc2VydCBhbGwoeCBpbiBzZWxmLmxhYmVscyBmb3IgeCBpbiAoJ2VudGFpbG1lbnQnLCduZXV0cmFsJywnY29udHJhZGljdGlvbicpKSxsYWJlbHMKICAgICAgICBzZWxmLnJldmlzaW9ucz1NT0RFTF9SRVZJU0lPTlMuY29weSgpCiAgICAgICAgc2VsZi5wcm9ncmVzcygnVjExIHJlYWR5IG9uICcrc2VsZi5kZXZpY2UsZmx1c2g9VHJ1ZSkKCiAgICBkZWYgc2F2ZV9jYWNoZShzZWxmKToKICAgICAgICBpZiBzZWxmLmNhY2hlX3BhdGg6CiAgICAgICAgICAgIHNlbGYuY2FjaGVfcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCiAgICAgICAgICAgIHRtcD1zZWxmLmNhY2hlX3BhdGgud2l0aF9zdWZmaXgoJy50bXAnKQogICAgICAgICAgICB0bXAud3JpdGVfdGV4dChqc29uLmR1bXBzKHNlbGYuY2FjaGUsZW5zdXJlX2FzY2lpPUZhbHNlKSkKICAgICAgICAgICAgdG1wLnJlcGxhY2Uoc2VsZi5jYWNoZV9wYXRoKQoKICAgIGRlZiB0cmFuc2xhdGUoc2VsZix0ZXh0cyk6CiAgICAgICAgbWlzc2luZz1saXN0KGRpY3QuZnJvbWtleXModCBmb3IgdCBpbiB0ZXh0cyBpZiAnbXRfbmxsYl9oYW5zX3Rlcm1zOicrZGlnZXN0KHQpIG5vdCBpbiBzZWxmLmNhY2hlKSkKICAgICAgICBmb3Igc3RhcnQgaW4gcmFuZ2UoMCxsZW4obWlzc2luZyksOCk6CiAgICAgICAgICAgIGJhdGNoPW1pc3Npbmdbc3RhcnQ6c3RhcnQrOF0KICAgICAgICAgICAgaW5wPXNlbGYubXRfdG9rKFt0cmFuc2xhdGlvbl9zb3VyY2UodCkgZm9yIHQgaW4gYmF0Y2hdLHJldHVybl90ZW5zb3JzPSdwdCcscGFkZGluZz1UcnVlLHRydW5jYXRpb249RmFsc2UpLnRvKHNlbGYuZGV2aWNlKQogICAgICAgICAgICBpZiBpbnBbJ2lucHV0X2lkcyddLnNoYXBlWzFdPjUxMjpyYWlzZSBWYWx1ZUVycm9yKCdUcmFuc2xhdGlvbiBzZWdtZW50IGV4Y2VlZHMgdG9rZW4gbGltaXQ7IG5vIHNpbGVudCB0cnVuY2F0aW9uLicpCiAgICAgICAgICAgIHdpdGggdG9yY2guaW5mZXJlbmNlX21vZGUoKToKICAgICAgICAgICAgICAgIG91dD1zZWxmLm10LmdlbmVyYXRlKCoqaW5wLG1heF9uZXdfdG9rZW5zPTMwMCxudW1fYmVhbXM9NCxmb3JjZWRfYm9zX3Rva2VuX2lkPXNlbGYubXRfdG9rLmNvbnZlcnRfdG9rZW5zX3RvX2lkcygiZW5nX0xhdG4iKSkKICAgICAgICAgICAgZm9yIHQsaWRzLHMgaW4gemlwKGJhdGNoLG91dCxzZWxmLm10X3Rvay5iYXRjaF9kZWNvZGUob3V0LHNraXBfc3BlY2lhbF90b2tlbnM9VHJ1ZSkpOgogICAgICAgICAgICAgICAgaWYgbGVuKGlkcyk+PTMwMCBhbmQgc2VsZi5tdC5jb25maWcuZW9zX3Rva2VuX2lkIG5vdCBpbiBpZHMudG9saXN0KClbMTpdOgogICAgICAgICAgICAgICAgICAgIHNlbGYuY2FjaGVbJ210X25sbGJfaGFuc190ZXJtczonK2RpZ2VzdCh0KV09eydlcnJvcic6J1RyYW5zbGF0aW9uIG91dHB1dCBsaW1pdCByZWFjaGVkOyB0aGUgcGFzc2FnZSBuZWVkcyBtYW51YWwgcmV2aWV3Lid9CiAgICAgICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgICAgIHNlbGYuY2FjaGVbJ210X25sbGJfaGFuc190ZXJtczonK2RpZ2VzdCh0KV09cwogICAgICAgICAgICBzZWxmLnNhdmVfY2FjaGUoKQogICAgICAgICAgICBpZiBzdGFydCU0MD09MDpzZWxmLnByb2dyZXNzKGYnVHJhbnNsYXRpb24gY3Jvc3MtY2hlY2s6IHttaW4oc3RhcnQrOCxsZW4obWlzc2luZykpfS97bGVuKG1pc3NpbmcpfSBuZXcgcGFzc2FnZXMnLGZsdXNoPVRydWUpCiAgICAgICAgcmV0dXJuIHt0OnNlbGYuY2FjaGVbJ210X25sbGJfaGFuc190ZXJtczonK2RpZ2VzdCh0KV0gZm9yIHQgaW4gdGV4dHN9CgogICAgZGVmIG5saV9zY29yZXMoc2VsZixwYWlycyk6CiAgICAgICAgZGVmIGtleShwKTpyZXR1cm4gJ25saTonK2RpZ2VzdChqc29uLmR1bXBzKHAsZW5zdXJlX2FzY2lpPUZhbHNlKSkKICAgICAgICBtaXNzaW5nPWxpc3QoZGljdC5mcm9ta2V5cyhwIGZvciBwIGluIHBhaXJzIGlmIGtleShwKSBub3QgaW4gc2VsZi5jYWNoZSkpCiAgICAgICAgZm9yIHN0YXJ0IGluIHJhbmdlKDAsbGVuKG1pc3NpbmcpLDEyKToKICAgICAgICAgICAgYmF0Y2g9bWlzc2luZ1tzdGFydDpzdGFydCsxMl0KICAgICAgICAgICAgaW5wPXNlbGYudG9rKFtwWzBdIGZvciBwIGluIGJhdGNoXSxbcFsxXSBmb3IgcCBpbiBiYXRjaF0scmV0dXJuX3RlbnNvcnM9J3B0JyxwYWRkaW5nPVRydWUsdHJ1bmNhdGlvbj1GYWxzZSkudG8oc2VsZi5kZXZpY2UpCiAgICAgICAgICAgIGlmIGlucFsnaW5wdXRfaWRzJ10uc2hhcGVbMV0+NTEyOnJhaXNlIFZhbHVlRXJyb3IoJ0luZmVyZW5jZSBzZWdtZW50IGV4Y2VlZHMgdG9rZW4gbGltaXQ7IG5vIHNpbGVudCB0cnVuY2F0aW9uLicpCiAgICAgICAgICAgIHdpdGggdG9yY2guaW5mZXJlbmNlX21vZGUoKTpzY29yZXM9c2VsZi5ubGkoKippbnApLmxvZ2l0cy5zb2Z0bWF4KC0xKS5jcHUoKS50b2xpc3QoKQogICAgICAgICAgICBmb3IgcCxzIGluIHppcChiYXRjaCxzY29yZXMpOnNlbGYuY2FjaGVba2V5KHApXT17azpzW2ldIGZvciBrLGkgaW4gc2VsZi5sYWJlbHMuaXRlbXMoKX0KICAgICAgICAgICAgc2VsZi5zYXZlX2NhY2hlKCkKICAgICAgICAgICAgaWYgc3RhcnQlMTIwPT0wOnNlbGYucHJvZ3Jlc3MoZidDb250ZXh0dWFsIGluZmVyZW5jZToge21pbihzdGFydCsxMixsZW4obWlzc2luZykpfS97bGVuKG1pc3NpbmcpfSBuZXcgcGFpcnMnLGZsdXNoPVRydWUpCiAgICAgICAgcmV0dXJuIHtwOnNlbGYuY2FjaGVba2V5KHApXSBmb3IgcCBpbiBwYWlyc30KCiAgICBkZWYgcHJlZGljdF9tYW55KHNlbGYsaW5wdXRzKToKICAgICAgICAjIE9ubHkgdGhlc2UgZmllbGRzIGNyb3NzIHRoZSBwcmVkaWN0b3IgYm91bmRhcnkuIFJlZmVyZW5jZXMgYXJlIG5vdCB1c2VkLgogICAgICAgIGNhc2VzPVt7azpyW2tdIGZvciBrIGluICgnY2FzZV9pZCcsJ2NoaW5lc2UnLCdlbmdsaXNoJyl9IGZvciByIGluIGlucHV0c10KICAgICAgICB3b3JrPVtdOyBpbnZhbGlkPXt9CiAgICAgICAgZm9yIGksciBpbiBlbnVtZXJhdGUoY2FzZXMpOgogICAgICAgICAgICB6LGU9bm9ybWFsKHJbJ2NoaW5lc2UnXSksbm9ybWFsKHJbJ2VuZ2xpc2gnXSkKICAgICAgICAgICAgaWYgbm90IHogb3Igbm90IGU6CiAgICAgICAgICAgICAgICBpbnZhbGlkW2ldPSdCb3RoIGxhbmd1YWdlIHZlcnNpb25zIGFyZSByZXF1aXJlZC4nO2NvbnRpbnVlCiAgICAgICAgICAgIGlmIG1heChsZW4oeiksbGVuKGUpKT42MDAwOgogICAgICAgICAgICAgICAgaW52YWxpZFtpXT0nVGhlIDYsMDAwLWNoYXJhY3RlciBpbnB1dCBsaW1pdCB3YXMgZXhjZWVkZWQuJztjb250aW51ZQogICAgICAgICAgICAjIFNob3J0IG5vdGljZXMgYXJlIGNvbXBhcmVkIGFzIGNvbXBsZXRlIGNvbnRleHRzLiBMb25nIG5vdGljZXMgYXJlCiAgICAgICAgICAgICMgZXhwbGljaXRseSB1bnJlc29sdmVkIHJhdGhlciB0aGFuIHNpbGVudGx5IGRyb3BwaW5nIGxhdGVyIHRleHQuCiAgICAgICAgICAgIGlmIGxlbihzZWxmLm10X3Rvayh0cmFuc2xhdGlvbl9zb3VyY2UoeikpWydpbnB1dF9pZHMnXSk+NDgwIG9yIGxlbihzZWxmLnRvayh6LGUpWydpbnB1dF9pZHMnXSk+NDgwOgogICAgICAgICAgICAgICAgaW52YWxpZFtpXT0nVGhpcyBub3RpY2UgZXhjZWVkcyBWMTHigJlzIHZhbGlkYXRlZCBjb250ZXh0IHdpbmRvdzsgc3BsaXQgaXQgaW50byBzaG9ydGVyIGNvbXBsZXRlIHNlY3Rpb25zIGZvciByZXZpZXcuJztjb250aW51ZQogICAgICAgICAgICB3b3JrLmFwcGVuZCgoaSx6LGUpKQogICAgICAgIHRleHRzPWxpc3QoZGljdC5mcm9ta2V5cyhbdCBmb3IgXyx6LGUgaW4gd29yayBmb3IgdCBpbiBbeixlLCpzZW50ZW5jZXMoeiksKnNlbnRlbmNlcyhlKV1dKSkKICAgICAgICB2ZWN0b3JzPXNlbGYubGFic2UuZW5jb2RlKHRleHRzLG5vcm1hbGl6ZV9lbWJlZGRpbmdzPVRydWUsc2hvd19wcm9ncmVzc19iYXI9RmFsc2UsYmF0Y2hfc2l6ZT0xNikgaWYgdGV4dHMgZWxzZSBbXQogICAgICAgIGVtYmVkZGluZ3M9ZGljdCh6aXAodGV4dHMsdmVjdG9ycykpCiAgICAgICAgc2VudGVuY2VfdHJhbnNsYXRpb25zPXNlbGYudHJhbnNsYXRlKGxpc3QoZGljdC5mcm9ta2V5cyh0IGZvciBfLHosXyBpbiB3b3JrIGZvciB0IGluIHNlbnRlbmNlcyh6KSkpKQogICAgICAgIHZhbGlkPVtdCiAgICAgICAgZm9yIGkseixlIGluIHdvcms6CiAgICAgICAgICAgIGlmIGFueShub3QgaXNpbnN0YW5jZShzZW50ZW5jZV90cmFuc2xhdGlvbnNbdF0sc3RyKSBmb3IgdCBpbiBzZW50ZW5jZXMoeikpOgogICAgICAgICAgICAgICAgaW52YWxpZFtpXT0nVGhlIHRyYW5zbGF0aW9uIGNyb3NzLWNoZWNrIGNvdWxkIG5vdCBjb3ZlciB0aGlzIG5vdGljZTsgbWFudWFsIHJldmlldyBpcyBuZWVkZWQuJwogICAgICAgICAgICBlbHNlOnZhbGlkLmFwcGVuZCgoaSx6LGUpKQogICAgICAgIHdvcms9dmFsaWQKICAgICAgICB0cmFuc2xhdGlvbnM9e3o6JyAnLmpvaW4oc2VudGVuY2VfdHJhbnNsYXRpb25zW3RdIGZvciB0IGluIHNlbnRlbmNlcyh6KSkgZm9yIF8seixfIGluIHdvcmt9CiAgICAgICAgdmFsaWQ9W10KICAgICAgICBmb3IgaSx6LGUgaW4gd29yazoKICAgICAgICAgICAgaWYgbGVuKHNlbGYudG9rKHRyYW5zbGF0aW9uc1t6XSxlKVsnaW5wdXRfaWRzJ10pPjUwMDoKICAgICAgICAgICAgICAgIGludmFsaWRbaV09J1RoZSB0cmFuc2xhdGVkIGNvbXBhcmlzb24gZXhjZWVkcyB0aGUgbW9kZWwgY29udGV4dDsgbWFudWFsIHJldmlldyBpcyBuZWVkZWQuJwogICAgICAgICAgICBlbHNlOnZhbGlkLmFwcGVuZCgoaSx6LGUpKQogICAgICAgIHdvcms9dmFsaWQKICAgICAgICBhbGlnbmVkPXtpOmFsaWduZWRfZ3JvdXBzKHNlbnRlbmNlcyh6KSxzZW50ZW5jZXMoZSksZW1iZWRkaW5ncykgZm9yIGkseixlIGluIHdvcmt9CiAgICAgICAgcXVhbGl0eV90ZXh0cz1saXN0KGRpY3QuZnJvbWtleXMoW3RyYW5zbGF0aW9uc1t6XSBmb3IgXyx6LF8gaW4gd29ya10rW3YgZm9yIHYgaW4gc2VudGVuY2VfdHJhbnNsYXRpb25zLnZhbHVlcygpIGlmIGlzaW5zdGFuY2UodixzdHIpXStbYSBmb3IgaSx6LGUgaW4gd29yayBmb3IgYSxiLHNpbSBpbiBhbGlnbmVkW2ldXStbJyAnLmpvaW4oc2VudGVuY2VfdHJhbnNsYXRpb25zW3RdIGZvciB0IGluIHNlbnRlbmNlcyh6KSBpZiB0IGluIGEpIGZvciBpLHosZSBpbiB3b3JrIGZvciBhLGIsc2ltIGluIGFsaWduZWRbaV1dKSkKICAgICAgICBxdWFsaXR5X3ZlY3RvcnM9c2VsZi5sYWJzZS5lbmNvZGUocXVhbGl0eV90ZXh0cyxub3JtYWxpemVfZW1iZWRkaW5ncz1UcnVlLHNob3dfcHJvZ3Jlc3NfYmFyPUZhbHNlLGJhdGNoX3NpemU9MTYpIGlmIHF1YWxpdHlfdGV4dHMgZWxzZSBbXQogICAgICAgIGVtYmVkZGluZ3MudXBkYXRlKGRpY3QoemlwKHF1YWxpdHlfdGV4dHMscXVhbGl0eV92ZWN0b3JzKSkpCiAgICAgICAgcGFpcnM9W10KICAgICAgICBmb3IgXyx6LGUgaW4gd29yazoKICAgICAgICAgICAgbXQ9dHJhbnNsYXRpb25zW3pdCiAgICAgICAgICAgIHBhaXJzLmV4dGVuZChbKHosZSksKGUseiksKG10LGUpLChlLG10KV0pCiAgICAgICAgICAgIGZvciBhLGIsc2ltIGluIGFsaWduZWRbX106CiAgICAgICAgICAgICAgICBhbT0nICcuam9pbihzZW50ZW5jZV90cmFuc2xhdGlvbnNbdF0gZm9yIHQgaW4gc2VudGVuY2VzKHopIGlmIHQgaW4gYSkKICAgICAgICAgICAgICAgIHBhaXJzLmV4dGVuZChbKGEsYiksKGIsYSksKGFtLGIpLChiLGFtKV0pCiAgICAgICAgc2NvcmVzPXNlbGYubmxpX3Njb3JlcyhwYWlycykKICAgICAgICBvdXRwdXQ9e30KICAgICAgICBmb3IgaSx6LGUgaW4gd29yazoKICAgICAgICAgICAgbXQ9dHJhbnNsYXRpb25zW3pdCiAgICAgICAgICAgIHA9W3Njb3Jlc1t4XSBmb3IgeCBpbiBbKHosZSksKGUseiksKG10LGUpLChlLG10KV1dCiAgICAgICAgICAgIHNpZ25hbD1kaWN0KGNyb3NzPVt4Wydjb250cmFkaWN0aW9uJ10gZm9yIHggaW4gcFs6Ml1dLGNyb3NzX2VudGFpbG1lbnQ9W3hbJ2VudGFpbG1lbnQnXSBmb3IgeCBpbiBwWzoyXV0sY3Jvc3NfbmV1dHJhbD1beFsnbmV1dHJhbCddIGZvciB4IGluIHBbOjJdXSwKICAgICAgICAgICAgICAgICAgICAgICAgdHJhbnNsYXRlZD1beFsnY29udHJhZGljdGlvbiddIGZvciB4IGluIHBbMjpdXSwKICAgICAgICAgICAgICAgICAgICAgICAgZW50YWlsbWVudD1beFsnZW50YWlsbWVudCddIGZvciB4IGluIHBbMjpdXSwKICAgICAgICAgICAgICAgICAgICAgICAgbmV1dHJhbD1beFsnbmV1dHJhbCddIGZvciB4IGluIHBbMjpdXSwKICAgICAgICAgICAgICAgICAgICAgICAgc2ltaWxhcml0eT1mbG9hdChlbWJlZGRpbmdzW3pdQGVtYmVkZGluZ3NbZV0pLHRyYW5zbGF0aW9uX3F1YWxpdHk9ZmxvYXQoZW1iZWRkaW5nc1t6XUBlbWJlZGRpbmdzW210XSksdHJhbnNsYXRpb249bXQpCiAgICAgICAgICAgIGZhY3RzPWRpcmVjdF9ldmlkZW5jZSh6LGUpCiAgICAgICAgICAgIGdyb3Vwcz1hbGlnbmVkW2ldCiAgICAgICAgICAgICMgUGFzc2FnZS1sZXZlbCBmYWN0cyBkZXRlY3Qgc3dhcHBlZCByb2xlcyBldmVuIHdoZW4gZG9jdW1lbnQtbGV2ZWwKICAgICAgICAgICAgIyB2YWx1ZSBzZXRzIGFncmVlLiBXZWFrIGFsaWdubWVudHMgZG8gbm90IGVzdGFibGlzaCBhIGZhY3R1YWwgZXJyb3IuCiAgICAgICAgICAgIGZvciBhLGIsc2ltIGluIGdyb3VwczoKICAgICAgICAgICAgICAgIGlmIHNpbT49c2VsZi5jb25maWdbJ2ZhY3RfYWxpZ25tZW50X21pbiddOgogICAgICAgICAgICAgICAgICAgIGZhY3RzLmV4dGVuZChkaXJlY3RfZXZpZGVuY2UoYSxiKSkKICAgICAgICAgICAgZmFjdHM9bGlzdCh7anNvbi5kdW1wcyh4LHNvcnRfa2V5cz1UcnVlLGVuc3VyZV9hc2NpaT1GYWxzZSk6eCBmb3IgeCBpbiBmYWN0c30udmFsdWVzKCkpCiAgICAgICAgICAgIHJlYXNvbnM9W10KICAgICAgICAgICAgZm9yIGYgaW4gZmFjdHM6CiAgICAgICAgICAgICAgICByZWFzb25zLmFwcGVuZChkaWN0KGtpbmQ9J2V4cGxpY2l0X2ZhY3QnLGNhdGVnb3J5PWZbJ2NhdGVnb3J5J10sCiAgICAgICAgICAgICAgICAgICAgd2h5PWYie2ZbJ2NhdGVnb3J5J119IGRpZmZlcnM6IEVuZ2xpc2gge2ZbJ2VuZ2xpc2hfdmFsdWUnXX0gW3tmWydlbmdsaXNoX3NwYW4nXX1dOyBDaGluZXNlIHtmWydjaGluZXNlX3ZhbHVlJ119IFt7ZlsnY2hpbmVzZV9zcGFuJ119XS4iLAogICAgICAgICAgICAgICAgICAgICoqe2s6diBmb3Igayx2IGluIGYuaXRlbXMoKSBpZiBrIT0nY2F0ZWdvcnknfSkpCiAgICAgICAgICAgIHBhc3NhZ2Vfc2lnbmFscz1bXQogICAgICAgICAgICBmb3IgYSxiLHNpbSBpbiBncm91cHM6CiAgICAgICAgICAgICAgICBhbT0nICcuam9pbihzZW50ZW5jZV90cmFuc2xhdGlvbnNbdF0gZm9yIHQgaW4gc2VudGVuY2VzKHopIGlmIHQgaW4gYSkKICAgICAgICAgICAgICAgIHBzPVtzY29yZXNbeF0gZm9yIHggaW4gWyhhLGIpLChiLGEpLChhbSxiKSwoYixhbSldXQogICAgICAgICAgICAgICAgc3M9ZGljdChjcm9zcz1beFsnY29udHJhZGljdGlvbiddIGZvciB4IGluIHBzWzoyXV0sY3Jvc3NfZW50YWlsbWVudD1beFsnZW50YWlsbWVudCddIGZvciB4IGluIHBzWzoyXV0sY3Jvc3NfbmV1dHJhbD1beFsnbmV1dHJhbCddIGZvciB4IGluIHBzWzoyXV0sZW50YWlsbWVudD1beFsnZW50YWlsbWVudCddIGZvciB4IGluIHBzWzI6XV0sbmV1dHJhbD1beFsnbmV1dHJhbCddIGZvciB4IGluIHBzWzI6XV0sdHJhbnNsYXRlZD1beFsnY29udHJhZGljdGlvbiddIGZvciB4IGluIHBzWzI6XV0sc2ltaWxhcml0eT1zaW0sdHJhbnNsYXRpb25fcXVhbGl0eT1mbG9hdChlbWJlZGRpbmdzW2FdQGVtYmVkZGluZ3NbYW1dKSxlbmdsaXNoX3NwYW49YixjaGluZXNlX3NwYW49YSx0cmFuc2xhdGlvbj1hbSkKICAgICAgICAgICAgICAgIHBhc3NhZ2Vfc2lnbmFscy5hcHBlbmQoc3MpCiAgICAgICAgICAgIHNpZ25hbFsncGFzc2FnZXMnXT1wYXNzYWdlX3NpZ25hbHMKICAgICAgICAgICAgb3V0cHV0W2ldPWRpY3QoY2FzZV9pZD1jYXNlc1tpXVsnY2FzZV9pZCddLGZsYWdnZWQ9Ym9vbChyZWFzb25zKSx3YXJuaW5ncz1yZWFzb25zLHNpZ25hbHM9c2lnbmFsLAogICAgICAgICAgICAgICAgZGVjaXNpb25fcGF0aD0ndjExX2NvbnRleHR1YWxfY29tcGFyaXNvbicsY292ZXJhZ2U9J2NvbXBsZXRlX25vdGljZScsdjdfZmFsbGJhY2s9RmFsc2UpCiAgICAgICAgZm9yIGksbWVzc2FnZSBpbiBpbnZhbGlkLml0ZW1zKCk6CiAgICAgICAgICAgIG91dHB1dFtpXT1kaWN0KGNhc2VfaWQ9Y2FzZXNbaV1bJ2Nhc2VfaWQnXSxmbGFnZ2VkPVRydWUsd2FybmluZ3M9W2RpY3Qoa2luZD0ndW5yZXNvbHZlZCcsY2F0ZWdvcnk9J2lucHV0IGNvdmVyYWdlJyx3aHk9bWVzc2FnZSxlbmdsaXNoX3NwYW49Y2FzZXNbaV1bJ2VuZ2xpc2gnXSxjaGluZXNlX3NwYW49Y2FzZXNbaV1bJ2NoaW5lc2UnXSldLGRlY2lzaW9uX3BhdGg9J3YxMV91bnJlc29sdmVkJyxjb3ZlcmFnZT0ndW5yZXNvbHZlZCcsdjdfZmFsbGJhY2s9RmFsc2UpCiAgICAgICAgcmV0dXJuIFtvdXRwdXRbaV0gZm9yIGkgaW4gcmFuZ2UobGVuKGNhc2VzKSldCgo=', 'facts.py': 'IiIiU21hbGwsIGNvbnRleHQtaW5kZXBlbmRlbnQgYmlsaW5ndWFsIGZhY3RzLiBOZXZlciBjb21wYXJlIGJhcmUgbmVnYXRpb24gY291bnRzLgoKT25seSBhIHVuaXF1ZWx5IHJlc29sdmVkIHByb3BlcnR5IGluIEJPVEggcGFzc2FnZXMgY2FuIGVzdGFibGlzaCBhIGNvbmZsaWN0LgpUaGVzZSBhcmUgY2hlY2tzLCBub3QgYSBjbGFpbSBvZiBleGhhdXN0aXZlIGxpbmd1aXN0aWMgY292ZXJhZ2UuCiIiIgppbXBvcnQgcmUKaW1wb3J0IHVuaWNvZGVkYXRhCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IGRlZmF1bHRkaWN0CmZyb20gZGVjaW1hbCBpbXBvcnQgRGVjaW1hbAoKCmRlZiBub3JtYWwodGV4dCk6CiAgICByZXR1cm4gdW5pY29kZWRhdGEubm9ybWFsaXplKCdORktDJywgdGV4dCkucmVwbGFjZSgn4oCZJywgIiciKS5zdHJpcCgpCgpDTiA9IGRpY3QoemlwKCfpm7bjgIfil4vkuIDkuozlhankuInlm5vkupTlha3kuIPlhavkuZ0nLCBbMCwwLDAsMSwyLDIsMyw0LDUsNiw3LDgsOV0pKQpOVSA9IHsn5Y2BJzoxMCwgJ+eZvic6MTAwLCAn5Y2DJzoxMDAwLCAn6JCsJzoxMDAwMH0KTlVNID0gcicoPzpcZCsoPzpbLC5dXGQrKSp8W+mbtuOAh+KXi+S4gOS6jOWFqeS4ieWbm+S6lOWFreS4g+WFq+S5neWNgeeZvuWNg+iQrF0rKScKRU5OVU0gPSBkaWN0KHppcCgnemVybyBvbmUgdHdvIHRocmVlIGZvdXIgZml2ZSBzaXggc2V2ZW4gZWlnaHQgbmluZSB0ZW4gZWxldmVuIHR3ZWx2ZSB0aGlydGVlbiBmb3VydGVlbiBmaWZ0ZWVuIHNpeHRlZW4gc2V2ZW50ZWVuIGVpZ2h0ZWVuIG5pbmV0ZWVuIHR3ZW50eScuc3BsaXQoKSxyYW5nZSgyMSkpKQoKZGVmIG51bWJlcihzKToKICAgIHM9cy5sb3dlcigpLnJlcGxhY2UoJywnLCcnKQogICAgaWYgcyBpbiBFTk5VTTpyZXR1cm4gRGVjaW1hbChFTk5VTVtzXSkKICAgIGlmIHJlLmZ1bGxtYXRjaChyJ1xkKyg/OlwuXGQrKT8nLHMpOnJldHVybiBEZWNpbWFsKHMpCiAgICBpZiBhbGwoeCBpbiBDTiBmb3IgeCBpbiBzKTpyZXR1cm4gRGVjaW1hbCgnJy5qb2luKHN0cihDTlt4XSkgZm9yIHggaW4gcykpCiAgICB0b3RhbD1jdXI9c2VjdGlvbj0wCiAgICBmb3IgeCBpbiBzOgogICAgICAgIGlmIHggaW4gQ046Y3VyPUNOW3hdCiAgICAgICAgZWxpZiB4IGluIE5VOgogICAgICAgICAgICBpZiBOVVt4XT09MTAwMDA6dG90YWwrPShzZWN0aW9uK2N1cikqMTAwMDA7c2VjdGlvbj1jdXI9MAogICAgICAgICAgICBlbHNlOnNlY3Rpb24rPShjdXIgb3IgMSkqTlVbeF07Y3VyPTAKICAgICAgICBlbHNlOnJhaXNlIFZhbHVlRXJyb3IocykKICAgIHJldHVybiBEZWNpbWFsKHRvdGFsK3NlY3Rpb24rY3VyKQoKIyBDbG9zZWQtdmFsdWUgY29uY2VwdHM7IHdvcmQtbGV2ZWwgYm91bmRhcmllcyBhdm9pZCBlLmcuICdyZWQnIGluc2lkZSAncmVxdWlyZWQnLgpDT05DRVBUUyA9IHsKICdjb2xvdXInOiBbKCdyZWQnLHIncmVkJyxyJ+e0heiJsicpLCgnYmx1ZScscidibHVlJyxyJ+iXjeiJsicpLCgnZ3JlZW4nLHInZ3JlZW4nLHIn57ag6ImyJyksKCd5ZWxsb3cnLHIneWVsbG93JyxyJ+m7g+iJsicpLCgnYmxhY2snLHInYmxhY2snLHIn6buR6ImyJyksKCd3aGl0ZScscid3aGl0ZScscifnmb3oibInKSwoJ29yYW5nZScscidvcmFuZ2UnLHIn5qmZ6ImyJyksKCdwdXJwbGUnLHIncHVycGxlJyxyJ+e0q+iJsicpXSwKICdsYW5ndWFnZSc6IFsoJ0VuZ2xpc2gnLHInRW5nbGlzaCcscifoi7Hmlod86Iux6KqeJyksKCdDaGluZXNlJyxyJ0NoaW5lc2UnLHIn5Lit5paHfOa8ouiqnicpLCgnQ2FudG9uZXNlJyxyJ0NhbnRvbmVzZScscifnsrXoqp585buj5p2x6KmxJyksKCdNYW5kYXJpbicscidNYW5kYXJpbnxQdXRvbmdodWEnLHIn5pmu6YCa6KmxfOWci+iqnicpLCgnSmFwYW5lc2UnLHInSmFwYW5lc2UnLHIn5pel5paHfOaXpeiqnicpLCgnS29yZWFuJyxyJ0tvcmVhbicscifpn5Pmlod86Z+T6KqeJyksKCdGcmVuY2gnLHInRnJlbmNoJyxyJ+azleaWh3zms5Xoqp4nKSwoJ1NwYW5pc2gnLHInU3BhbmlzaCcscifopb/nj63niZnmlod86KW/54+t54mZ6KqeJyldLAogJ29yaWVudGF0aW9uJzogWygncG9ydHJhaXQnLHIncG9ydHJhaXQnLHIn55u05ZCRfOe4seWQkScpLCgnbGFuZHNjYXBlJyxyJ2xhbmRzY2FwZScscifmqavlkJEnKV0sCiAnbWF0ZXJpYWwnOiBbKCdnbGFzcycscidnbGFzcycscifnjrvnkoMnKSwoJ3BsYXN0aWMnLHIncGxhc3RpYycscifloZHohqB85aGR5paZfOiGoOaovScpLCgnbWV0YWwnLHInbWV0YWwnLHIn6YeR5bGsJyksKCdhbHVtaW5pdW0nLHInYWx1bWluKD86aXVtfHVtKScscifpi4EnKSwoJ3dvb2QnLHInd29vZCg/OmVuKT8nLHIn5pyo6KO9fOacqOadkCcpLCgncGFwZXInLHIncGFwZXInLHIn57SZ6KO9fOe0meW8tScpXSwKICdmaWxlIGZvcm1hdCc6IFsoeCwgeCwgeCkgZm9yIHggaW4gWydQREYnLCdDU1YnLCdET0NYJywnWExTWCcsJ0pQRUcnLCdQTkcnLCdaSVAnXV0sCiAnd2Vla2RheSc6IFsoZSxlK3Incz8nLHInKD865pif5pyffOmAsXzlkagpJyt6KSBmb3IgZSx6IGluIHppcChbJ01vbmRheScsJ1R1ZXNkYXknLCdXZWRuZXNkYXknLCdUaHVyc2RheScsJ0ZyaWRheScsJ1NhdHVyZGF5JywnU3VuZGF5J10sWyfkuIAnLCfkuownLCfkuIknLCflm5snLCfkupQnLCflha0nLCdb5pel5aSpXSddKV0sCiAnZnJlcXVlbmN5JzogWygnZGFpbHknLHInZGFpbHl8ZXZlcnkgZGF5JyxyJ+avj+aXpXzmr4/lpKknKSwoJ3dlZWtseScscid3ZWVrbHl8ZXZlcnkgd2Vlaycscifmr4/pgLF85q+P5pif5pyffOavj+WRqCcpLCgnbW9udGhseScscidtb250aGx5fGV2ZXJ5IG1vbnRoJyxyJ+avj+aciCcpLCgncXVhcnRlcmx5JyxyJ3F1YXJ0ZXJseXxldmVyeSB0aHJlZSBtb250aHMnLHIn5q+P5a2jfOavj+S4ieWAi+aciCcpLCgneWVhcmx5JyxyJ3llYXJseXxhbm51YWxseXxhbm51YWx8ZXZlcnkgeWVhcicscifmr4/lubR85LiA5bm05LiA5bqmJyldLAogJ2RvY3VtZW50IGZvcm0nOiBbKCdvcmlnaW5hbCcscidvcmlnaW5hbCg/OnMpPycscifmraPmnKx85Y6f5Lu2JyksKCdjb3B5JyxyJ2NvcCg/Onl8aWVzKXxwaG90b2NvcCg/Onl8aWVzKScsciflia/mnKx85b2x5Y2w5pysJyldLAp9CgpkZWYgZXZpZGVuY2UoY2F0ZWdvcnksIHosIGUpOgogICAgcmV0dXJuIGRpY3QoY2F0ZWdvcnk9Y2F0ZWdvcnksY2hpbmVzZV92YWx1ZT1zdHIoelswXSksZW5nbGlzaF92YWx1ZT1zdHIoZVswXSksY2hpbmVzZV9zcGFuPXpbMV0sZW5nbGlzaF9zcGFuPWVbMV0pCgpkZWYgdW5pcXVlKGhpdHMpOgogICAgdmFsdWVzPXt4WzBdIGZvciB4IGluIGhpdHN9CiAgICByZXR1cm4gaGl0c1swXSBpZiBsZW4odmFsdWVzKT09MSBlbHNlIE5vbmUKCgpkZWYgY29uY2VwdF9mYWN0cyh0ZXh0LCBsYW5nKToKICAgIGZvdW5kPXt9CiAgICBmb3IgY2F0ZWdvcnksIGVudHJpZXMgaW4gQ09OQ0VQVFMuaXRlbXMoKToKICAgICAgICBoaXRzPVtdCiAgICAgICAgZm9yIHZhbHVlLGVwLHpwIGluIGVudHJpZXM6CiAgICAgICAgICAgIHBhdD1yJ1xiKD86JytlcCtyJylcYicgaWYgbGFuZz09J2VuJyBlbHNlIHpwCiAgICAgICAgICAgIGhpdHMuZXh0ZW5kKCh2YWx1ZSxtWzBdKSBmb3IgbSBpbiByZS5maW5kaXRlcihwYXQsdGV4dCxyZS5JKSkKICAgICAgICBmPXVuaXF1ZShoaXRzKQogICAgICAgIGlmIGY6Zm91bmRbY2F0ZWdvcnldPWYKICAgIHJldHVybiBmb3VuZAoKIyBRdWFudGl0eSB1bml0cyBpbmNsdWRlIGEgZGltZW5zaW9uIGFuZCBjb252ZXJzaW9uIG11bHRpcGxpZXIuIENvcnJlY3QgY2hhbmdlcwojIHN1Y2ggYXMgMSBtIC8gMTAwIGNtIGFyZSBlcXVpdmFsZW50OyBwZXJjZW50YWdlcyBpbiDnmb7liIbmr5QgYXJlIG5vdCB0aGUgbnVtYmVyIDEwMC4KVU5JVFM9WwogKCdsZW5ndGgnLCcxJyxyJ20oPzpldHJlfGV0ZXIpcz98bScscifnsbN85YWs5bC6JyksCiAoJ2xlbmd0aCcsJy4wMScscidjZW50aW0oPzpldHJlfGV0ZXIpcz98Y20nLHIn5Y6Y57GzfOWFrOWIhicpLAogKCdsZW5ndGgnLCcuMDAxJyxyJ21pbGxpbSg/OmV0cmV8ZXRlcilzP3xtbScscifmr6vnsbMnKSwKICgnbGVuZ3RoJywnMTAwMCcscidraWxvbSg/OmV0cmV8ZXRlcilzP3xrbScsciflhazph4x85Y2D57GzJyksCiAoJ21hc3MnLCcxJyxyJ2tpbG9ncmFtcz98a2cnLHIn5YWs5pakfOWNg+WFiycpLCgnbWFzcycsJy4wMDEnLHInZ3JhbXM/fGcnLHIn5YWLfOWFrOWFiycpLAogKCdkdXJhdGlvbicsJzEnLHInbWludXRlcz98bWlucz8nLHIn5YiG6ZCYJyksKCdkdXJhdGlvbicsJzYwJyxyJ2hvdXJzP3xocnM/JyxyJ+Wwj+aZgicpLAogKCdkYXlzJywnMScscidkYXlzPycsciflpKl85pelJyksKCd3ZWVrcycsJzEnLHInd2Vla3M/JyxyJ+aYn+acn3zpgLF85ZGoJyksCiAoJ2FnZScsJzEnLHIneWVhcnM/IG9sZHx5ZWFycz8gb2YgYWdlJyxyJ+atsicpLAogKCdhbW91bnQgSEtEJywnMScscidIS0R8SEsgZG9sbGFycz98SG9uZyBLb25nIGRvbGxhcnM/JyxyJ+a4r+WFgycpLAogKCdwZXJjZW50YWdlJywnMScsciclfHBlcmNlbnR8cGVyIGNlbnQnLHInJXzlt7Tku5knKSwKICgncGFnZXMnLCcxJyxyJ3BhZ2VzPycscifpoIEnKSwoJ2Zsb29ycycsJzEnLHInZmxvb3JzPycscifmqJMnKSwKICgnb2NjdXJyZW5jZXMnLCcxJyxyJ3RpbWVzPycscifmrKEnKSwKICgnY291bnQnLCcxJyxyJ3Blb3BsZXxwZXJzb25zP3xwdXBpbHM/fHN0dWRlbnRzP3xwYXJ0aWNpcGFudHM/fGNvcGllc3xpdGVtcz98Zm9ybXM/JyxyJ+S6unzlkI185YCLfOS7vScpLApdCk1PTlRIUz17eDppKzEgZm9yIGkseCBpbiBlbnVtZXJhdGUoJ0phbnVhcnkgRmVicnVhcnkgTWFyY2ggQXByaWwgTWF5IEp1bmUgSnVseSBBdWd1c3QgU2VwdGVtYmVyIE9jdG9iZXIgTm92ZW1iZXIgRGVjZW1iZXInLnNwbGl0KCkpfQoKZGVmIHF1YW50aXR5X2ZhY3RzKHRleHQsIGxhbmcpOgogICAgcmVzdWx0PWRlZmF1bHRkaWN0KGxpc3QpCiAgICB0PW5vcm1hbCh0ZXh0KQogICAgbnVtPU5VTSBpZiBsYW5nPT0nemgnIGVsc2UgcicoPzpcZCsoPzpbLC5dXGQrKSp8JysgJ3wnLmpvaW4oRU5OVU0pK3InKScKICAgIGZvciBkaW1lbnNpb24sbXVsdCxlcCx6cCBpbiBVTklUUzoKICAgICAgICBwYXQ9KHJmJyh7bnVtfSlccyooPzp7enB9KSg/IVthLXpdKScgaWYgbGFuZz09J3poJyBlbHNlIHJmJ1xiKHtudW19KVxzKig/OntlcH0pKD8hW2Etel0pJykKICAgICAgICBmb3IgbSBpbiByZS5maW5kaXRlcihwYXQsdCxyZS5JKToKICAgICAgICAgICAgIyDkuozmrKEgLyDkuIDoiKwgLyDnmb7liIbmr5QgY2Fubm90IGJlIG1pc3Rha2VuIGZvciBudW1lcmljIHF1YW50aXRpZXMuCiAgICAgICAgICAgIHJlc3VsdFtkaW1lbnNpb25dLmFwcGVuZCgoc3RyKG51bWJlcihtWzFdKSpEZWNpbWFsKG11bHQpKSxtWzBdKSkKICAgIGlmIGxhbmc9PSdlbic6CiAgICAgICAgZm9yIG0gaW4gcmUuZmluZGl0ZXIocidcYkhLXCRccyooXGQrKD86WywuXVxkKykqKScsdCxyZS5JKTpyZXN1bHRbJ2Ftb3VudCBIS0QnXS5hcHBlbmQoKHN0cihudW1iZXIobVsxXSkpLG1bMF0pKQogICAgICAgIGZvciBtIGluIHJlLmZpbmRpdGVyKHInXGIoXGR7MSwyfSkoPzpzdHxuZHxyZHx0aCk/XHMrKCcrJ3wnLmpvaW4oTU9OVEhTKStyJylcYicsdCxyZS5JKTpyZXN1bHRbJ2RhdGUnXS5hcHBlbmQoKGYne01PTlRIU1ttWzJdLnRpdGxlKCldfS17aW50KG1bMV0pfScsbVswXSkpCiAgICAgICAgZm9yIG0gaW4gcmUuZmluZGl0ZXIocidcYignKyd8Jy5qb2luKE1PTlRIUykrcicpXHMrKFxkezEsMn0pKD86c3R8bmR8cmR8dGgpP1xiJyx0LHJlLkkpOnJlc3VsdFsnZGF0ZSddLmFwcGVuZCgoZid7TU9OVEhTW21bMV0udGl0bGUoKV19LXtpbnQobVsyXSl9JyxtWzBdKSkKICAgIGVsc2U6CiAgICAgICAgZm9yIG0gaW4gcmUuZmluZGl0ZXIocmYnKHtOVU19KeaciCh7TlVNfSlb5pel6JmfXScsdCk6cmVzdWx0WydkYXRlJ10uYXBwZW5kKChmJ3tudW1iZXIobVsxXSl9LXtudW1iZXIobVsyXSl9JyxtWzBdKSkKICAgICAgICAjIERhdGVzIGFyZSBub3QgZHVyYXRpb25zLiBBIGRhdGUncyDml6UgaXMgcmVtb3ZlZCBmcm9tIGRheSBxdWFudGl0aWVzLgogICAgICAgIHJlc3VsdFsnZGF5cyddPVtxIGZvciBxIGluIHJlc3VsdFsnZGF5cyddIGlmIG5vdCByZS5zZWFyY2gocifmnIgnK3JlLmVzY2FwZShxWzFdKSx0KV0KICAgICMgRGVjaW1hbHMgYXJlIGNhbm9uaWNhbGlzZWQgZm9yIGVxdWFsaXR5IGFmdGVyIFNJIGNvbnZlcnNpb24uCiAgICBmb3IgY2F0ZWdvcnksIGhpdHMgaW4gcmVzdWx0Lml0ZW1zKCk6CiAgICAgICAgaWYgY2F0ZWdvcnkhPSdkYXRlJzpyZXN1bHRbY2F0ZWdvcnldPVsoc3RyKERlY2ltYWwodikubm9ybWFsaXplKCkpLHMpIGZvciB2LHMgaW4gaGl0c10KICAgIHJldHVybiBkaWN0KHJlc3VsdCkKCgpkZWYgZGlyZWN0X2V2aWRlbmNlKGNoaW5lc2UsIGVuZ2xpc2gpOgogICAgeixlPW5vcm1hbChjaGluZXNlKSxub3JtYWwoZW5nbGlzaCkKICAgIG91dD1bXQogICAgZm9yIGV4dHJhY3RvciBpbiAoY29uY2VwdF9mYWN0cyxxdWFudGl0eV9mYWN0cyk6CiAgICAgICAgemYsZWY9ZXh0cmFjdG9yKHosJ3poJyksZXh0cmFjdG9yKGUsJ2VuJykKICAgICAgICBmb3IgY2F0ZWdvcnkgaW4gc29ydGVkKHpmLmtleXMoKSAmIGVmLmtleXMoKSk6CiAgICAgICAgICAgIGEsYj16ZltjYXRlZ29yeV0sZWZbY2F0ZWdvcnldCiAgICAgICAgICAgIGlmIGV4dHJhY3Rvcj09cXVhbnRpdHlfZmFjdHM6CiAgICAgICAgICAgICAgICAjIFNldHMgb2YgdmFsdWVzIGNhbm5vdCBhc3NpZ24gc2VtYW50aWMgcm9sZXM7IGRlZmVyIHJlcGVhdGVkCiAgICAgICAgICAgICAgICAjIGRpc3RpbmN0IHF1YW50aXRpZXMgdG8gY29udGV4dHVhbCBpbmZlcmVuY2UgcmF0aGVyIHRoYW4gZ3Vlc3MuCiAgICAgICAgICAgICAgICBhLGI9dW5pcXVlKGEpLHVuaXF1ZShiKQogICAgICAgICAgICBpZiBhIGFuZCBiIGFuZCBhWzBdIT1iWzBdOm91dC5hcHBlbmQoZXZpZGVuY2UoY2F0ZWdvcnksYSxiKSkKICAgIGZvciBjYXRlZ29yeSxwYXR0ZXJuIGluIFsKICAgICAgICAoJ2VtYWlsIGFkZHJlc3MnLHInW1x3ListXStAW1x3Li1dK1wuW2EtekEtWl17Mix9JyksCiAgICAgICAgKCdVUkwgZGVzdGluYXRpb24nLHInaHR0cHM/Oi8vW15cczw+77yM44CC77yb77yJ44CNXSsnKSwKICAgICAgICAoJ3JlZmVyZW5jZSBpZGVudGlmaWVyJyxyJyg/PCFbQS1aYS16XGRdKVtBLVpdezEsNX1bLS9dXGR7Mix9KD86Wy0vXVxkKyk/JyldOgogICAgICAgIHpzPVsobVswXS5yc3RyaXAoJy4sOyknKSxtWzBdKSBmb3IgbSBpbiByZS5maW5kaXRlcihwYXR0ZXJuLHopXQogICAgICAgIGVzPVsobVswXS5yc3RyaXAoJy4sOyknKSxtWzBdKSBmb3IgbSBpbiByZS5maW5kaXRlcihwYXR0ZXJuLGUpXQogICAgICAgIGEsYj11bmlxdWUoenMpLHVuaXF1ZShlcykKICAgICAgICBpZiBhIGFuZCBiIGFuZCBhWzBdIT1iWzBdOm91dC5hcHBlbmQoZXZpZGVuY2UoY2F0ZWdvcnksYSxiKSkKICAgIHpjLGVjPWNsb2NrX2ZhY3RzKHosJ3poJyksY2xvY2tfZmFjdHMoZSwnZW4nKQogICAgYSxiPXVuaXF1ZSh6YyksdW5pcXVlKGVjKQogICAgaWYgYSBhbmQgYiBhbmQgYVswXSE9YlswXTpvdXQuYXBwZW5kKGV2aWRlbmNlKCdjbG9jayB0aW1lJyxhLGIpKQogICAgb3V0LmV4dGVuZChib3VuZGFyeV9ldmlkZW5jZSh6LGUpKQogICAgb3V0LmV4dGVuZChtb2RhbGl0eV9ldmlkZW5jZSh6LGUpKQogICAgcmV0dXJuIG91dAoKCmRlZiBib3VuZGFyeV9ldmlkZW5jZSh6LGUpOgogICAgb3V0PVtdCiAgICAjIFBvc2l0aW9uIHdvcmRzIG9ubHkgd2hlbiB1c2VkIHNwYXRpYWxseSwgbm90IHRoZSBwYXN0IHRlbnNlIG9mIGxlYXZlLgogICAgZW5fZGlyZWN0aW9ucz1bKHYscicoPzooPzp0b3x0b3dhcmRzP3xwb2ludHM/fHBvaW50aW5nfGZhY2luZ3x0dXJuKVxzKyg/OnRoZVxzKyk/JytwYXQrcid8JytwYXQrcidbLSBdaGFuZCknKSBmb3IgdixwYXQgaW4gWygnbGVmdCcsJ2xlZnQnKSwoJ3JpZ2h0JywncmlnaHQnKSwoJ3VwJywndXB3YXJkcz98dXAnKSwoJ2Rvd24nLCdkb3dud2FyZHM/fGRvd24nKV1dCiAgICB6aF9kaXJlY3Rpb25zPVsoJ2xlZnQnLHIn5bemKD865pa5fOWBtHzpgoopJyksKCdyaWdodCcsciflj7MoPzrmlrl85YG0fOmCiiknKSwoJ3VwJyxyJ+S4iig/OuaWuXzlgbR86YKKKScpLCgnZG93bicscifkuIsoPzrmlrl85YG0fOmCiiknKV0KICAgIHpoPXVuaXF1ZShbKHYsbVswXSkgZm9yIHYscCBpbiB6aF9kaXJlY3Rpb25zIGZvciBtIGluIHJlLmZpbmRpdGVyKHAseildKQogICAgZW49dW5pcXVlKFsodixtWzBdKSBmb3IgdixwIGluIGVuX2RpcmVjdGlvbnMgZm9yIG0gaW4gcmUuZmluZGl0ZXIocCxlLHJlLkkpXSkKICAgIGlmIHpoIGFuZCBlbiBhbmQgemhbMF0hPWVuWzBdOm91dC5hcHBlbmQoZXZpZGVuY2UoJ3NwYXRpYWwgZGlyZWN0aW9uJyx6aCxlbikpCiAgICAjIFVuaXQtb25seSBpbnN0cnVjdGlvbnMgY2FuIGNoYW5nZSBldmVuIHdoZW4gdGhlcmUgaXMgbm8gbnVtZXJpYyB2YWx1ZS4KICAgIGZvciBkaW0gaW4gWydsZW5ndGgnLCdtYXNzJ106CiAgICAgICAgenM9W107ZXM9W10KICAgICAgICBmb3IgZCxtdWx0LGVwLHpwIGluIFVOSVRTOgogICAgICAgICAgICBpZiBkIT1kaW06Y29udGludWUKICAgICAgICAgICAgenMuZXh0ZW5kKChtdWx0LG1bMF0pIGZvciBtIGluIHJlLmZpbmRpdGVyKHInKD865LulfOeUqHzllq7kvY1b54K65pivOu+8ml0pKD86Jyt6cCtyJyknLHopKQogICAgICAgICAgICBlcy5leHRlbmQoKG11bHQsbVswXSkgZm9yIG0gaW4gcmUuZmluZGl0ZXIocidcYig/OmlufHVzaW5nKVxzKyg/OicrZXArcicpXGInLGUscmUuSSkpCiAgICAgICAgYSxiPXVuaXF1ZSh6cyksdW5pcXVlKGVzKQogICAgICAgIGlmIGEgYW5kIGIgYW5kIGFbMF0hPWJbMF06b3V0LmFwcGVuZChldmlkZW5jZShkaW0rJyB1bml0JyxhLGIpKQogICAgIyBJbmNsdXNpdmUgLyBleGNsdXNpdmUgYm91bmRhcmllcyBtdXN0IHJlZmVyIHRvIHRoZSBzYW1lIGV4cGxpY2l0IHZhbHVlLgogICAgem49TlVNO2VuPXInXGQrKD86XC5cZCspPycKICAgIHpocGF0dGVybnM9WygnPj0nLHJmJ+S4jeWwkeaWvCh7em59KXzmnIDlsJEoe3pufSl8KHt6bn0pKD865q2yKT/miJbku6XkuIonKSwoJz4nLHJmJ+i2hemBjih7em59KXzpq5jmlrwoe3pufSknKSwoJzw9JyxyZifkuI3lpJrmlrwoe3pufSl85pyA5aSaKHt6bn0pfCh7em59KSg/Ouatsik/5oiW5Lul5LiLJyksKCc8JyxyZiflsJHmlrwoe3pufSl85pyq5ru/KHt6bn0pfOS9juaWvCh7em59KScpXQogICAgZW5wYXR0ZXJucz1bKCc+PScscmYnXGJhdCBsZWFzdCAoe2VufSl8XGIoe2VufSkoPzogeWVhcnM/KT8gb3IgKD86bW9yZXxvdmVyfGFib3ZlKScpLCgnPicscmYnXGJtb3JlIHRoYW4gKHtlbn0pfFxiYWJvdmUgKHtlbn0pJyksKCc8PScscmYnXGJhdCBtb3N0ICh7ZW59KXxcYm5vIG1vcmUgdGhhbiAoe2VufSl8XGIoe2VufSkoPzogeWVhcnM/KT8gb3IgKD86bGVzc3x1bmRlcnxiZWxvdyknKSwoJzwnLHJmJ1xibGVzcyB0aGFuICh7ZW59KXxcYnVuZGVyICh7ZW59KXxcYmJlbG93ICh7ZW59KScpXQogICAgZGVmIGxpbWl0cyhwcyx0KToKICAgICAgICBmb3VuZD1bXQogICAgICAgIGZvciBvcCxwYXQgaW4gcHM6CiAgICAgICAgICAgIGZvciBtIGluIHJlLmZpbmRpdGVyKHBhdCx0LHJlLkkpOmZvdW5kLmFwcGVuZCgoKG9wLHN0cihudW1iZXIobmV4dChnIGZvciBnIGluIG0uZ3JvdXBzKCkgaWYgZykpKSksbVswXSkpCiAgICAgICAgcmV0dXJuIHVuaXF1ZShmb3VuZCkKICAgIGEsYj1saW1pdHMoemhwYXR0ZXJucyx6KSxsaW1pdHMoZW5wYXR0ZXJucyxlKQogICAgaWYgYSBhbmQgYiBhbmQgYVswXVsxXT09YlswXVsxXSBhbmQgYVswXVswXSE9YlswXVswXTpvdXQuYXBwZW5kKGV2aWRlbmNlKCdpbmNsdXNpdmUvZXhjbHVzaXZlIGJvdW5kYXJ5JyxhLGIpKQogICAgcmV0dXJuIG91dAoKIyBPcHRpb25hbCB2ZXJzdXMgcmVxdWlyZWQgaXMgbWF0ZXJpYWwgZXZlbiB3aGVuIG5laXRoZXIgc2lkZSBpcyBhIGxvZ2ljYWwKIyBjb250cmFkaWN0aW9uLiBPbmx5IGF0dGFjaCBhbiBleHBsaWNpdCBtb2RhbCBpbW1lZGlhdGVseSBiZWZvcmUgYSBzaGFyZWQgYWN0aW9uLgpBQ1RJT05TID0gewogJ3N1Ym1pdCc6IChyJ3N1Ym1pdHxsb2RnZXxoYW5kIGluJyxyJ+aPkOS6pHzpgZ7kuqR85Lqk5ZueJyksCiAnYnJpbmcnOiAocidicmluZycscifmlJzluLZ85pSc5ZCMfOW4tuWCmXzluLblkIwnKSwKICdzaG93JzogKHInc2hvd3xwcmVzZW50fHByb2R1Y2UnLHIn5Ye656S6JyksCiAncGF5JzogKHIncGF5JyxyJ+e5s+S7mHzmlK/ku5h857mz5LqkJyksCiAncmVnaXN0ZXInOiAocidyZWdpc3RlcnxlbnJvbHxlbnJvbGwnLHIn55m76KiYfOWgseWQjScpLAogJ2F0dGVuZCc6IChyJ2F0dGVuZCcsciflh7rluK0nKSwKICd1cGxvYWQnOiAocid1cGxvYWQnLHIn5LiK6LyJfOS4iuWCsycpLAogJ3NpZ24nOiAocidzaWduJyxyJ+ewvee9sicpLAogJ3JldGFpbic6IChyJ2tlZXB8cmV0YWluJyxyJ+S/neeVmScpLAogJ2NvbGxlY3QnOiAocidjb2xsZWN0JyxyJ+mgmOWPlnzlj5bku7YnKSwKICdyZXBseSc6IChyJ3JlcGx5fHJlc3BvbmQnLHIn5Zue6KaGfOWbnuW+qScpLAogJ3NlbmQnOiAocidzZW5kfGZvcndhcmQnLHIn55m86YCBfOWvhOmAgScpLAogJ3VzZSc6IChyJ3VzZScscifkvb/nlKgnKSwKICdlbnRlcic6IChyJ2VudGVyJyxyJ+mAsuWFpScpLAogJ2NhbmNlbCc6IChyJ2NhbmNlbCcsciflj5bmtognKSwKICdyZXBsYWNlJzogKHIncmVwbGFjZScscifmm7Tmj5t85Y+W5LujJyksCn0KCmRlZiBtb2RhbGl0eV9ldmlkZW5jZSh6LGUpOgogICAgIyBVbnJlc29sdmVkIGNvbmRpdGlvbmFsIHNjb3BlIGlzIGxlZnQgdG8gY29udGV4dHVhbCBpbmZlcmVuY2UsIG5vdCByZWR1Y2VkCiAgICAjIHRvIGEgZG9jdW1lbnQtd2lkZSBiYWcgb2Ygb2JsaWdhdGlvbiB0b2tlbnMuCiAgICBpZiByZS5zZWFyY2gocidcYig/OmlmfHVubGVzc3xvbmx5fGV4Y2VwdHx3aGV0aGVyKVxiJyxlLHJlLkkpIG9yIHJlLnNlYXJjaChyJ+Wmgnzoi6V86Zmk6Z2efOWPquaciXzlg4V85omNfOmZpOWklicseik6cmV0dXJuIFtdCiAgICBvdXQ9W10KICAgIGVtPVsoJ29wdGlvbmFsJyxyJyg/Om5lZWQgbm90fGRvKD86ZXMpPyBub3QgKD86bmVlZHxoYXZlKSB0b3woPzppc3xhcmUpIG5vdCByZXF1aXJlZCB0bylccyokJyksCiAgICAgICAgKCdwcm9oaWJpdGVkJyxyJyg/Om11c3Qgbm90fG1heSBub3R8Y2Fubm90fGNhblx4Mjd0fCg/OmlzfGFyZSkgbm90IGFsbG93ZWQgdG8pXHMqJCcpLAogICAgICAgICgncmVxdWlyZWQnLHInKD88IW5vdCApKD86bXVzdHxuZWVkIHRvfGhhdmUgdG98aGFzIHRvfCg/OmlzfGFyZSkgcmVxdWlyZWQgdG8pXHMqJCcpLAogICAgICAgICgncGVybWl0dGVkJyxyJyg/Om1heXxjYW58KD86aXN8YXJlKSBhbGxvd2VkIHRvKVxzKiQnKV0KICAgIHptPVsoJ29wdGlvbmFsJyxyJyg/OueEoemgiHzmr4vpoIh85LiN5b+FfOS4jeeUqClccyokJyksKCdwcm9oaWJpdGVkJyxyJyg/OuS4jeW+l3zkuI3lj69856aB5q2ifOS4jeWHhilccyokJyksCiAgICAgICAgKCdyZXF1aXJlZCcscicoPzwhW+avi+eEoeS4jV0pKD865b+F6aCIfOmgiClccyokJyksKCdwZXJtaXR0ZWQnLHInKD88IVvkuI1dKSg/OuWPr+S7pXzlj68pXHMqJCcpXQogICAgZGVmIGZhY3QodCxhcCxwYXR0ZXJucyxsYW5nKToKICAgICAgICBhY3Rpb25zPWxpc3QocmUuZmluZGl0ZXIocidcYig/OicrYXArcicpXGInIGlmIGxhbmc9PSdlbicgZWxzZSBhcCx0LHJlLkkpKQogICAgICAgIGlmIGxlbihhY3Rpb25zKSE9MTpyZXR1cm4gTm9uZQogICAgICAgIGE9YWN0aW9uc1swXTtwcmVmaXg9dFttYXgoMCxhLnN0YXJ0KCktNjApOmEuc3RhcnQoKV0KICAgICAgICBmb3IgdmFsdWUscGF0IGluIHBhdHRlcm5zOgogICAgICAgICAgICBtPXJlLnNlYXJjaChwYXQscHJlZml4LHJlLkkpCiAgICAgICAgICAgIGlmIG06cmV0dXJuIHZhbHVlLG1bMF0rYVswXQogICAgICAgIHJldHVybiBOb25lCiAgICBmb3IgbmFtZSwoZXAsenApIGluIEFDVElPTlMuaXRlbXMoKToKICAgICAgICBhLGI9ZmFjdCh6LHpwLHptLCd6aCcpLGZhY3QoZSxlcCxlbSwnZW4nKQogICAgICAgICMgT3B0aW9uYWwgdnMgcGVybWl0dGVkIGNhbiBiZSBlcXVpdmFsZW50OyBuZWl0aGVyIGVzdGFibGlzaGVzIGFuIG9ibGlnYXRpb24uCiAgICAgICAgaWYgYSBhbmQgYiBhbmQgYVswXSE9YlswXSBhbmQgbm90IHthWzBdLGJbMF19PD17J29wdGlvbmFsJywncGVybWl0dGVkJ306CiAgICAgICAgICAgIG91dC5hcHBlbmQoZXZpZGVuY2UoJ29ibGlnYXRpb24gdG8gJytuYW1lLGEsYikpCiAgICByZXR1cm4gb3V0CgoKZGVmIGNsb2NrX2ZhY3RzKHRleHQsbGFuZyk6CiAgICBmb3VuZD1bXTtvY2N1cGllZD1bXQogICAgaWYgbGFuZz09J2VuJzoKICAgICAgICBmb3IgbSBpbiByZS5maW5kaXRlcihyJ1xiKFxkezEsMn0pKD86WzouXShcZHsyfSkpP1xzKihbYXBdKVwuP21cLj8nLHRleHQscmUuSSk6CiAgICAgICAgICAgIGgsbW49aW50KG1bMV0pLGludChtWzJdIG9yIDApCiAgICAgICAgICAgIGlmIDE8PWg8PTEyIGFuZCBtbjw2MDoKICAgICAgICAgICAgICAgIGZvdW5kLmFwcGVuZCgoc3RyKChoJTEyKygxMiBpZiBtWzNdLmxvd2VyKCk9PSdwJyBlbHNlIDApKSo2MCttbiksbVswXSkpO29jY3VwaWVkLmFwcGVuZChtLnNwYW4oKSkKICAgICAgICBmb3IgdmFsLHAgaW4gWyg3MjAscidcYm5vb25cYicpLCgwLHInXGJtaWRuaWdodFxiJyldOgogICAgICAgICAgICBmb3VuZC5leHRlbmQoKHN0cih2YWwpLG1bMF0pIGZvciBtIGluIHJlLmZpbmRpdGVyKHAsdGV4dCxyZS5JKSkKICAgIGVsc2U6CiAgICAgICAgZm9yIG0gaW4gcmUuZmluZGl0ZXIocmYnKOS4iuWNiHzkuIvljYh85pma5LiKfOWHjOaZqHzkuK3ljYgpKHtOVU19KSg/OuaZgnzpu558OikoPzooe05VTX0p5YiGPyk/Jyx0ZXh0KToKICAgICAgICAgICAgaCxtbj1pbnQobnVtYmVyKG1bMl0pKSxpbnQobnVtYmVyKG1bM10pKSBpZiBtWzNdIGVsc2UgMAogICAgICAgICAgICBpZiAwPD1oPD0xMiBhbmQgbW48NjA6CiAgICAgICAgICAgICAgICBmb3VuZC5hcHBlbmQoKHN0cigoaCUxMisoMTIgaWYgbVsxXSBpbiBbJ+S4i+WNiCcsJ+aZmuS4iicsJ+S4reWNiCddIGVsc2UgMCkpKjYwK21uKSxtWzBdKSk7b2NjdXBpZWQuYXBwZW5kKG0uc3BhbigpKQogICAgICAgIGZvciB2YWwscCBpbiBbKDcyMCxyJ+ato+WNiCcpLCgwLHIn5Y2I5aScJyldOmZvdW5kLmV4dGVuZCgoc3RyKHZhbCksbVswXSkgZm9yIG0gaW4gcmUuZmluZGl0ZXIocCx0ZXh0KSkKICAgIGZvciBtIGluIHJlLmZpbmRpdGVyKHInKD88IVxkKShcZHsxLDJ9KTooXGR7Mn0pKD8hXGQpJyx0ZXh0KToKICAgICAgICBpZiBhbnkobS5zdGFydCgpPGIgYW5kIG0uZW5kKCk+YSBmb3IgYSxiIGluIG9jY3VwaWVkKTpjb250aW51ZQogICAgICAgIGgsbW49aW50KG1bMV0pLGludChtWzJdKQogICAgICAgIGlmIGg8MjQgYW5kIG1uPDYwOmZvdW5kLmFwcGVuZCgoc3RyKGgqNjArbW4pLG1bMF0pKQogICAgcmV0dXJuIGZvdW5kCg==', 'terminology.py': 'IiIiQm91bmRlZCB0ZXJtaW5vbG9neSBmb3IgdHJhbnNsYXRpb24gY3Jvc3MtY2hlY2tzLCBpbmRlcGVuZGVudCBvZiBub3RpY2Ugd29yZGluZy4KClRoaXMgaXMgYSBnbG9zc2FyeSwgbm90IGFuIGVxdWl2YWxlbmNlIHZlcmRpY3Qgb3IgYSBzdXBwb3J0ZWQtbm90aWNlIHdoaXRlbGlzdC4KVGVybXMgbm90IGxpc3RlZCBjb250aW51ZSB0aHJvdWdoIG5vcm1hbCBjb250ZXh0dWFsIHRyYW5zbGF0aW9uLiBObyBiZW5jaG1hcmsKaWRlbnRpZmllcnMsIHRhcmdldCB3b3JkaW5nLCBsYWJlbHMgb3IgZ29sZCByYXRpb25hbGVzIGVudGVyIHRoaXMgc3RlcC4KIiIiCmltcG9ydCByZQpmcm9tIG9wZW5jYyBpbXBvcnQgT3BlbkNDCl9UT19TSU1QTElGSUVEPU9wZW5DQygidDJzIikKClRFUk1TID0gewogICAgJ+amleaouSc6J2JhbnlhbiB0cmVlJywgJ+aon+aouSc6J2NhbXBob3IgdHJlZScsICfmtIvntKvojYonOidvcmNoaWQgdHJlZScsCiAgICAn55u45oCd5qi5JzonYWNhY2lhIHRyZWUnLCAn5p2+5qi5JzoncGluZSB0cmVlJywgJ+apoeaouSc6J29hayB0cmVlJywKICAgICfmpZPmqLknOidtYXBsZSB0cmVlJywgJ+ajlearmuaouSc6J3BhbG0gdHJlZScsICfmnKjmo4nmqLknOidjb3R0b24gdHJlZScsCiAgICAn5p+z5qi5Jzond2lsbG93IHRyZWUnLCAn6YqA5p2P5qi5JzonZ2lua2dvIHRyZWUnLAogICAgJ+ijneijsSc6J21vdW50aW5nIGFydHdvcmsnLAogICAgJ+ebtOWQkeagvOW8jyc6J3BvcnRyYWl0IGZvcm1hdCcsICfnuLHlkJHmoLzlvI8nOidwb3J0cmFpdCBmb3JtYXQnLAogICAgJ+apq+WQkeagvOW8jyc6J2xhbmRzY2FwZSBmb3JtYXQnLAogICAgJ+avq+exsyc6J21pbGxpbWV0cmVzJywgJ+WOmOexsyc6J2NlbnRpbWV0cmVzJywgJ+WFrOaWpCc6J2tpbG9ncmFtcycsCiAgICAn5riv5YWDJzonSG9uZyBLb25nIGRvbGxhcnMnLCAn5bel5L2c5aSpJzond29ya2luZyBkYXlzJywgJ+abhuaXpSc6J2NhbGVuZGFyIGRheXMnLAogICAgJ+ato+acrCc6J29yaWdpbmFsIGRvY3VtZW50JywgJ+W9seWNsOacrCc6J3Bob3RvY29weScsCiAgICAn5qWt5Li756uL5qGI5rOV5ZyYJzoib3duZXJzJyBjb3Jwb3JhdGlvbiIsICfmpa3kuLvms5XlnJgnOiJvd25lcnMnIGNvcnBvcmF0aW9uIiwKICAgICfpnZ7niZ/liKknOidub25wcm9maXQnLCAn56eB6ZqxJzoncHJpdmFjeScsICfpm7vpg7UnOidlbWFpbCcsCn0KX1BBVFRFUk49cmUuY29tcGlsZSgnfCcuam9pbihtYXAocmUuZXNjYXBlLHNvcnRlZChURVJNUyxrZXk9bGVuLHJldmVyc2U9VHJ1ZSkpKSkKCmRlZiB0cmFuc2xhdGlvbl9zb3VyY2UoY2hpbmVzZSk6CiAgICAjIEFuY2hvciBrbm93biB0ZWNobmljYWwgdGVybXMgYmVmb3JlIGdlbmVyYXRpb24sIHNvIGEgbWlzdHJhbnNsYXRlZCByYXJlCiAgICAjIHdvcmQgZG9lcyBub3QgbWFzcXVlcmFkZSBhcyBkaXNhZ3JlZW1lbnQgd2l0aCB0aGUgc3VwcGxpZWQgRW5nbGlzaC4KICAgIHJldHVybiBfVE9fU0lNUExJRklFRC5jb252ZXJ0KF9QQVRURVJOLnN1YihsYW1iZGEgbTonICcrVEVSTVNbbVswXV0rJyAnLGNoaW5lc2UpKQo=', 'backbone_config.json': 'ewogICJmYWN0X2FsaWdubWVudF9taW4iOiAwLjYKfQo=', 'features.py': 'IiIiTGFuZ3VhZ2UtbW9kZWwgc2lnbmFsIGZlYXR1cmVzLCB3aXRob3V0IGNhc2UgSURzLCB0b3BpY3Mgb3IgcmVmZXJlbmNlIGxhYmVscy4iIiIKaW1wb3J0IG51bXB5IGFzIG5wClBST0JTPSgnY3Jvc3MnLCdjcm9zc19lbnRhaWxtZW50JywnY3Jvc3NfbmV1dHJhbCcsJ3RyYW5zbGF0ZWQnLCdlbnRhaWxtZW50JywnbmV1dHJhbCcpCgpkZWYgcGFzc2FnZV92ZWN0b3Iocyk6CiAgICBwPW5wLmFycmF5KFt2IGZvciBrIGluIFBST0JTIGZvciB2IGluIHNba11dLGR0eXBlPWZsb2F0KQogICAgbHA9bnAubG9nKG5wLmNsaXAocCwxZS01LDEtMWUtNSkvbnAuY2xpcCgxLXAsMWUtNSwxKSkKICAgIHN1bW1hcmllcz1bXQogICAgZm9yIGsgaW4gUFJPQlM6CiAgICAgICAgYSxiPXNba107c3VtbWFyaWVzLmV4dGVuZChbbWluKGEsYiksbWF4KGEsYiksYWJzKGEtYildKQogICAgcmV0dXJuIG5wLnJfW3AsbHAsc3VtbWFyaWVzLFtzWydzaW1pbGFyaXR5J10sc1sndHJhbnNsYXRpb25fcXVhbGl0eSddXV0KCmRlZiB2ZWN0b3IocHJlZGljdGlvbik6CiAgICBzPXByZWRpY3Rpb25bJ3NpZ25hbHMnXTthPW5wLmFycmF5KFtwYXNzYWdlX3ZlY3RvcihwKSBmb3IgcCBpbiBzWydwYXNzYWdlcyddXSkKICAgIGRpcmVjdD1hbnkod1sna2luZCddPT0nZXhwbGljaXRfZmFjdCcgZm9yIHcgaW4gcHJlZGljdGlvblsnd2FybmluZ3MnXSkKICAgIHJldHVybiBucC5yX1twYXNzYWdlX3ZlY3RvcihzKSxhLm1pbigwKSxhLm1heCgwKSxhLm1lYW4oMCksbGVuKGEpLGludChkaXJlY3QpXQoKCmRlZiBmZWF0dXJlX25hbWVzKCk6CiAgICB1bml0PVtmJ3twfS57ZGlyZWN0aW9ufScgZm9yIHAgaW4gUFJPQlMgZm9yIGRpcmVjdGlvbiBpbiBbJ2ZvcndhcmQnLCdyZXZlcnNlJ11dCiAgICB1bml0Kz1bJ2xvZ2l0LicreCBmb3IgeCBpbiB1bml0LmNvcHkoKV0KICAgIHVuaXQrPVtmJ3twfS57c3RhdH0nIGZvciBwIGluIFBST0JTIGZvciBzdGF0IGluIFsnbWluJywnbWF4JywnYXN5bW1ldHJ5J11dCiAgICB1bml0Kz1bJ3NpbWlsYXJpdHknLCd0cmFuc2xhdGlvbl9xdWFsaXR5J10KICAgIHJldHVybiBbZid7Z3JvdXB9LntuYW1lfScgZm9yIGdyb3VwIGluIFsnd2hvbGUnLCdwYXNzYWdlX21pbicsJ3Bhc3NhZ2VfbWF4JywncGFzc2FnZV9tZWFuJ10gZm9yIG5hbWUgaW4gdW5pdF0rWydwYXNzYWdlX2NvdW50JywnZGlyZWN0X2ZhY3QnXQo=', 'coverage_rules.py': 'IiIiQ29ycm9ib3JhdGVkIG9uZS13YXkgaW5mb3JtYXRpb24gbG9zcyB3aXRoIGFuIGV4cGxpY2l0IGV4dHJhIHF1YWxpZmllci4KCk9wZXJhdG9yIHByZXNlbmNlIGlzIG5ldmVyIGVub3VnaCB0byBmbGFnIGEgcGFpci4gQm90aCBvcmlnaW5hbCBiaWxpbmd1YWwgTkxJCmFuZCB0cmFuc2xhdGVkIE5MSSBtdXN0IHN1cHBvcnQgdGhlIHNhbWUgZGlyZWN0aW9uIG9mIGluZm9ybWF0aW9uIGxvc3MuCiIiIgppbXBvcnQgcmUKCk9QRVJBVE9SUyA9IFsKICAgICgnZXhjZXB0aW9uJywgcidcYig/OnVubGVzc3xleGNlcHR8ZXhjbHVkaW5nfGV4ZW1wdClcYicsIHIn6Zmk6Z2efOmZpC57MCwxOH1b5aSW5LulXXzosYHlhY185L6L5aSWfOS4jeWMheaLrCcpLAogICAgKCdhZGRpdGlvbmFsIHJlcXVpcmVtZW50JywgcidcYig/OmFuZHxhbG9uZyB3aXRofGFzIHdlbGwgYXN8aW5jbHVkaW5nfHdpdGgpXGInLCByJ+WPinzlkox86IiHfOS4pnzpgKPlkIx85YyF5ousfOmZhOaciScpLApdCgoKZGVmIHVuc3VwcG9ydGVkX2RldGFpbHMoc2lnbmFsLCBlbmdsaXNoLCBjaGluZXNlKToKICAgIGZvdW5kID0gW10KICAgIGlmIHNpZ25hbFsndHJhbnNsYXRpb25fcXVhbGl0eSddIDwgLjc6CiAgICAgICAgcmV0dXJuIGZvdW5kCiAgICBmb3IgZGlyZWN0aW9uIGluICgwLCAxKToKICAgICAgICBpZiBub3QgKHNpZ25hbFsnY3Jvc3NfbmV1dHJhbCddW2RpcmVjdGlvbl0gPj0gLjk1CiAgICAgICAgICAgICAgICBhbmQgc2lnbmFsWyduZXV0cmFsJ11bZGlyZWN0aW9uXSA+PSAuOTUKICAgICAgICAgICAgICAgIGFuZCBzaWduYWxbJ2Nyb3NzX2VudGFpbG1lbnQnXVsxLWRpcmVjdGlvbl0gPj0gLjg1CiAgICAgICAgICAgICAgICBhbmQgc2lnbmFsWydlbnRhaWxtZW50J11bMS1kaXJlY3Rpb25dID49IC44NSk6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgZm9yIGNhdGVnb3J5LCBlbl9wYXR0ZXJuLCB6aF9wYXR0ZXJuIGluIE9QRVJBVE9SUzoKICAgICAgICAgICAgZW0gPSByZS5zZWFyY2goZW5fcGF0dGVybiwgZW5nbGlzaCwgcmUuSSkKICAgICAgICAgICAgem0gPSByZS5zZWFyY2goemhfcGF0dGVybiwgY2hpbmVzZSkKICAgICAgICAgICAgaWYgbm90ICgoZGlyZWN0aW9uID09IDAgYW5kIGVtIGFuZCBub3Qgem0pIG9yCiAgICAgICAgICAgICAgICAgICAgKGRpcmVjdGlvbiA9PSAxIGFuZCB6bSBhbmQgbm90IGVtKSk6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBtYXRjaCA9IGVtIGlmIGRpcmVjdGlvbiA9PSAwIGVsc2Ugem0KICAgICAgICAgICAgdGV4dCA9IGVuZ2xpc2ggaWYgZGlyZWN0aW9uID09IDAgZWxzZSBjaGluZXNlCiAgICAgICAgICAgIGRldGFpbCA9IHJlLnNwbGl0KHInW+OAgu+8ge+8n118KD88PVsuIT9dKVxzKycsIHRleHRbbWF0Y2guc3RhcnQoKTpdKVswXQogICAgICAgICAgICBsYW5ndWFnZSA9ICdFbmdsaXNoJyBpZiBkaXJlY3Rpb24gPT0gMCBlbHNlICdDaGluZXNlJwogICAgICAgICAgICBmb3VuZC5hcHBlbmQoZGljdChraW5kPSd1bnN1cHBvcnRlZF9kZXRhaWwnLCBjYXRlZ29yeT1jYXRlZ29yeSwKICAgICAgICAgICAgICAgIHdoeT1mJ3tsYW5ndWFnZX0gaW5jbHVkZXMg4oCce2RldGFpbH3igJ0uIEJvdGggc2VtYW50aWMgY29tcGFyaXNvbnMgaW5kaWNhdGUgdGhhdCB0aGlzIGRldGFpbCBpcyBub3Qgc3VwcG9ydGVkIGJ5IHRoZSBvdGhlciB2ZXJzaW9uOyBjaGVjayBmb3IgYW4gb21pdHRlZCBvciBhZGRlZCBjb25kaXRpb24uJywKICAgICAgICAgICAgICAgIGRldGFpbF9sYW5ndWFnZT1sYW5ndWFnZSwgdW5zdXBwb3J0ZWRfZGV0YWlsPWRldGFpbCwKICAgICAgICAgICAgICAgIGVuZ2xpc2hfc3Bhbj1lbmdsaXNoLCBjaGluZXNlX3NwYW49Y2hpbmVzZSwKICAgICAgICAgICAgICAgIG1hY2hpbmVfYmFja3RyYW5zbGF0aW9uPXNpZ25hbFsndHJhbnNsYXRpb24nXSwKICAgICAgICAgICAgICAgIGV2aWRlbmNlX2RpcmVjdGlvbj1kaXJlY3Rpb24pKQogICAgcmV0dXJuIGZvdW5kCg==', 'learned_head.json': '...', 'run.py': 'IiIiUnVuIHRoZSBmcm96ZW4gcHJlZGljdG9yIG9uIGFuIGlucHV0cy1vbmx5IENTVjsgbmV2ZXIgc2NvcmUgZHVyaW5nIGluZmVyZW5jZS4iIiIKaW1wb3J0IGFyZ3BhcnNlLGNzdixqc29uLGhhc2hsaWIsc3lzLHRpbWUscGxhdGZvcm0saW1wb3J0bGliLm1ldGFkYXRhLGlvCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIGRhdGV0aW1lIGltcG9ydCBkYXRldGltZSx0aW1lem9uZQpmcm9tIGVuZ2luZSBpbXBvcnQgQ2hlY2tlcixNT0RFTF9JRFMKcD1hcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCnAuYWRkX2FyZ3VtZW50KCctLWlucHV0cycscmVxdWlyZWQ9VHJ1ZSk7cC5hZGRfYXJndW1lbnQoJy0tb3V0cHV0JyxyZXF1aXJlZD1UcnVlKTtwLmFkZF9hcmd1bWVudCgnLS1jYWNoZScscmVxdWlyZWQ9VHJ1ZSkKcC5hZGRfYXJndW1lbnQoJy0tYmF0Y2gtc2l6ZScsdHlwZT1pbnQsZGVmYXVsdD01MCk7cC5hZGRfYXJndW1lbnQoJy0tcmVxdWlyZS1mcm96ZW4nLGFjdGlvbj0nc3RvcmVfdHJ1ZScpCmE9cC5wYXJzZV9hcmdzKCk7cm9vdD1QYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50CnNvdXJjZV9maWxlcz1bJ19faW5pdF9fLnB5JywnZW5naW5lLnB5JywnYmFja2JvbmUucHknLCdmYWN0cy5weScsJ3Rlcm1pbm9sb2d5LnB5JywnYmFja2JvbmVfY29uZmlnLmpzb24nLCdmZWF0dXJlcy5weScsJ2NvdmVyYWdlX3J1bGVzLnB5JywnbGVhcm5lZF9oZWFkLmpzb24nLCdydW4ucHknXQpoYXNoZXM9e2Y6aGFzaGxpYi5zaGEyNTYoKHJvb3QvZikucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKSBmb3IgZiBpbiBzb3VyY2VfZmlsZXN9CmlmIGEucmVxdWlyZV9mcm96ZW46CiBtYW5pZmVzdD1qc29uLmxvYWRzKChyb290Lydmcm96ZW5fbWFuaWZlc3QuanNvbicpLnJlYWRfdGV4dCgpKTthc3NlcnQgaGFzaGVzPT1tYW5pZmVzdFsnZmlsZXMnXSwnRnJvemVuIGNoZWNrZXIgY2hhbmdlZCcKcmF3PVBhdGgoYS5pbnB1dHMpLnJlYWRfYnl0ZXMoKQpyb3dzPWxpc3QoY3N2LkRpY3RSZWFkZXIoaW8uU3RyaW5nSU8ocmF3LmRlY29kZSgndXRmLTgtc2lnJykpKSkKYXNzZXJ0IHNldChyb3dzWzBdKT09eydjYXNlX2lkJywnZW5nbGlzaCcsJ2NoaW5lc2UnfSwnUHJlZGljdGlvbiBpbnB1dCBtdXN0IGNvbnRhaW4gbm8gZ29sZCBsYWJlbHMgb3IgcmF0aW9uYWxlcycKYXNzZXJ0IGxlbih7clsnY2FzZV9pZCddIGZvciByIGluIHJvd3N9KT09bGVuKHJvd3MpCnN0YXJ0ZWQ9ZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0YykuaXNvZm9ybWF0KCkKY2hlY2tlcj1DaGVja2VyKGNhY2hlPWEuY2FjaGUpCnJlc3VsdHM9W107b3V0cHV0PVBhdGgoYS5vdXRwdXQpO291dHB1dC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCnBhcnRpYWw9b3V0cHV0LndpdGhfc3VmZml4KCcuanNvbmwnKTtwYXJ0aWFsLndyaXRlX3RleHQoJycpCmZvciBpIGluIHJhbmdlKDAsbGVuKHJvd3MpLGEuYmF0Y2hfc2l6ZSk6CiBiYXRjaD1jaGVja2VyLnByZWRpY3RfbWFueShyb3dzW2k6aSthLmJhdGNoX3NpemVdKTtyZXN1bHRzLmV4dGVuZChiYXRjaCkKIHdpdGggcGFydGlhbC5vcGVuKCdhJykgYXMgZjoKICBmb3IgciBpbiBiYXRjaDpmLndyaXRlKGpzb24uZHVtcHMocixlbnN1cmVfYXNjaWk9RmFsc2UpKydcbicpCiBwcmludCgnUFJFRElDVElPTlMnLGxlbihyZXN1bHRzKSwnLycsbGVuKHJvd3MpLGZsdXNoPVRydWUpCm1ldGFkYXRhPWRpY3QodmVyc2lvbj0nVjExJyxzdGFydGVkPXN0YXJ0ZWQsY29tcGxldGVkPWRhdGV0aW1lLm5vdyh0aW1lem9uZS51dGMpLmlzb2Zvcm1hdCgpLAogaW5wdXRzX3NoYTI1Nj1oYXNobGliLnNoYTI1NihyYXcpLmhleGRpZ2VzdCgpLGZpbGVzPWhhc2hlcyxtb2RlbF9pZHM9TU9ERUxfSURTLAogbW9kZWxfcmV2aXNpb25zPWNoZWNrZXIucmV2aXNpb25zLGRldmljZT1jaGVja2VyLmRldmljZSxweXRob249cGxhdGZvcm0ucHl0aG9uX3ZlcnNpb24oKSwKIHBhY2thZ2VzPXtwOmltcG9ydGxpYi5tZXRhZGF0YS52ZXJzaW9uKHApIGZvciBwIGluIFsndG9yY2gnLCd0cmFuc2Zvcm1lcnMnLCdzZW50ZW5jZS10cmFuc2Zvcm1lcnMnLCdzZW50ZW5jZXBpZWNlJywncGFuZGFzJywnc2NpcHknLCdudW1weScsJ29wZW5jYy1weXRob24tcmVpbXBsZW1lbnRlZCddfSwKIHByZWRpY3Rpb25zPWxlbihyZXN1bHRzKSx2N19mYWxsYmFja19jYWxscz0wKQpvdXRwdXQud3JpdGVfdGV4dChqc29uLmR1bXBzKGRpY3QobWV0YWRhdGE9bWV0YWRhdGEscmVzdWx0cz1yZXN1bHRzKSxlbnN1cmVfYXNjaWk9RmFsc2UsaW5kZW50PTIpKQpwcmludCgnVjExIENPTVBMRVRFJyxsZW4ocmVzdWx0cyksZmx1c2g9VHJ1ZSkK', 'frozen_manifest.json': 'ewogICJ2ZXJzaW9uIjogIlYxMSIsCiAgImZyb3plbl9hdCI6ICIyMDI2LTExLTE1VDAzOjM4OjI4LjM0NDAyNCswMDowMCIsCiAgImZpbGVzIjogewogICAgIl9faW5pdF9fLnB5IjogIjQ0NGZkYTg4YmZkY2EyMzE2MmYxMzM1ZWQ5YTcxYzYzZWRhODAyYzc0ZmE4NDMwYThlYjE2OTkzOTI1YWQ2ODgiLAogICAgImVuZ2luZS5weSI6ICI1MDhmNmI5OTY0YzdjYjNiMzIyMjZhMmU2MjAyNTg1YzljMGY1NzI0M2YzYzhlNTMwMDYxMDI2NzEwOTBiNzVkIiwKICAgICJiYWNrYm9uZS5weSI6ICIzNGQ0MWYyMDlhMDZkZWRmOGQ0M2VmZWU3ZWM1Njk0ZDg5MGI4MjFlMzE5MjI1NGJkMjA4ZDI4YTZjMDgwMDdjIiwKICAgICJmYWN0cy5weSI6ICJjY2JmM2EwNTc4MWZlZGE3OTc0YjJiNzk2MDAzMjEyYmM0MWNlNGU1ZmUwYmVkNTAzZWIzNDgyNTRhMjllNzhmIiwKICAgICJ0ZXJtaW5vbG9neS5weSI6ICIzZjExYTU4MTY5ZGUyYTgxZmNhZjM4NjQ3OWYzZmY4NjQxZDM5M2U4ODQ0MWE3ODEyYTNlZGNlMWZlMWRlMTkwIiwKICAgICJiYWNrYm9uZV9jb25maWcuanNvbiI6ICJmZDJkMDZhZDE2NGI3OWMyZmQyZTMzZTM1MzhiMDg3N2M4YWQxZDczMDI0OTI5NzE5M2UxMTc1YTZhYjAwZDZiIiwKICAgICJmZWF0dXJlcy5weSI6ICJkMmM2ZGY2ZjFjNGQ0ZWEyMGI0NDZlODQyMjI4M2U0NWU0ZWNjZGJjODU4ZWRlYWYxN2JlYjEwN2JiYTRiMDEyIiwKICAgICJjb3ZlcmFnZV9ydWxlcy5weSI6ICIwNmUyMDZmZGI2YWY4ZjQ1NGM2NGExMTY5NWQ0NTQzZTcxNzBhZGZiMTI3ZmE3NDFiMDcxZTg1ZjQwNDVhNDg2IiwKICAgICJsZWFybmVkX2hlYWQuanNvbiI6ICI4ZTkwMDcwYTkyNzYzMjAxOTgxNjgyNzY5NWE1ZGY2YzAzMzJhODcwODQxMWY5YmNkNzE2MTZmYmU3YzE1OTFmIiwKICAgICJydW4ucHkiOiAiNzc2NDI3NTQ0MzZhYzljNmEwZmYyMTU3YTQ0YmYzMTk5NjZhYWU5N2Y3YzM2M2Y5ZTdmZjk0YmJmN2ViNTVjMCIKICB9LAogICJzb3VyY2VfYnVuZGxlX3NoYTI1NiI6ICJkNzg0OTQxYmNkYzkxYmI5ZWY0MDdlZDljNWYyYjUyYmYxY2I0Y2ZhZjY3M2IwNzcyMzA0ZWEzZDE4YjY1M2VhIiwKICAibW9kZWxfcmV2aXNpb25zIjogewogICAgImVtYmVkZGluZyI6ICI4MzYxMjFhMDUzM2U1NjY0YjIxYzdhYWNjNWQyMjk1MWYyYjhiMjViIiwKICAgICJubGkiOiAiYjUxMTNlYjM4YWI2M2VmZGQ3ZjI4MGY4YzE0NGVhOGIxM2Y5NzhjZSIsCiAgICAidHJhbnNsYXRpb24iOiAiZjhkMzMzYTA5OGQxOWI0ZmQ5YThiMThmOTQxNzA0ODdhZDNmODIxZCIKICB9LAogICJtb2RlbF9pZHMiOiB7CiAgICAiZW1iZWRkaW5nIjogInNlbnRlbmNlLXRyYW5zZm9ybWVycy9MYUJTRSIsCiAgICAibmxpIjogIk1vcml0ekxhdXJlci9tRGVCRVJUYS12My1iYXNlLXhubGktbXVsdGlsaW5ndWFsLW5saS0ybWlsNyIsCiAgICAidHJhbnNsYXRpb24iOiAiZmFjZWJvb2svbmxsYi0yMDAtZGlzdGlsbGVkLTYwME0iCiAgfSwKICAiZGV2ZWxvcG1lbnRfZGF0YSI6IFsKICAgICJob2xkb3V0X3Y5L3Y5X3Vuc2Vlbl8xMDAwLmNzdiIsCiAgICAiaG9sZG91dF92MTAvdjEwX2JsaW5kXzEwMDAuY3N2IgogIF0sCiAgInRyYWluaW5nX3JlY29yZHNfc2hhMjU2IjogIjhlNTM1MTJhN2Q0YjZmM2Y1MWZkN2VjNjBmOTg3NzkxZjYyNjJjM2Y3N2EzOWZkNTkwZGFkZTVkY2NhMDhmYjUiLAogICJ0cmFpbmluZ19wcm9jZWR1cmUiOiAicmVndWxhcml6ZWQgbG9naXN0aWMgaGVhZCBvbiAyMDAwIG9sZCBjYXNlczsgNSBmYW1pbHktZGlzam9pbnQgZm9sZHMgZm9yIG1vZGVsIGFuZCB0aHJlc2hvbGQgc2VsZWN0aW9uOyBleHRyYSBjb3Jyb2JvcmF0ZWQgcXVhbGlmaWVyLWxvc3MgcnVsZSIsCiAgImRlY2lzaW9uX3RocmVzaG9sZCI6IDAuNzg1LAogICJDIjogMC4wMSwKICAiaG9sZG91dF90ZXh0X29yX2xhYmVsc19yZWFkIjogZmFsc2UsCiAgIm5vX2Z1cnRoZXJfdHVuaW5nIjogdHJ1ZSwKICAic21va2VfY2FzZXMiOiAxMiwKICAicHl0aG9uIjogIjMuMTEuMTQiLAogICJwYWNrYWdlcyI6IHsKICAgICJ0b3JjaCI6ICIyLjE1LjAiLAogICAgInRyYW5zZm9ybWVycyI6ICI1LjIyLjEiLAogICAgInNlbnRlbmNlLXRyYW5zZm9ybWVycyI6ICI1LjcuMCIsCiAgICAic2VudGVuY2VwaWVjZSI6ICIwLjIuMyIsCiAgICAicGFuZGFzIjogIjMuMS4xIiwKICAgICJzY2lweSI6ICIxLjE3LjEiLAogICAgIm51bXB5IjogIjIuNC42IiwKICAgICJzY2lraXQtbGVhcm4iOiAiMS45LjEiLAogICAgIm9wZW5jYy1weXRob24tcmVpbXBsZW1lbnRlZCI6ICIwLjEuNyIKICB9Cn0K'}
for name,data in payload.items():
    (folder/name).write_bytes(base64.b64decode(data))
manifest=json.loads((folder/'frozen_manifest.json').read_text())
for name,digest in manifest['files'].items():
    assert hashlib.sha256((folder/name).read_bytes()).hexdigest()==digest
sys.path.insert(0,'/content')
from v11_checker.engine import Checker
checker=Checker(cache='/content/v11_cache.json')
check_notice=checker.check_notice
print('Frozen V11 loaded. No V7 fallback.')


ModuleNotFoundError: No module named 'backbone'

In [ ]:
chinese_text='請於星期五提交表格。'
english_text='Please submit the form on Friday.'
check_notice(chinese_text, english_text)


In [ ]:
# Upload the INPUTS-ONLY CSV supplied in the benchmark bundle.
# Its three columns are case_id, english, chinese; keep the answer key separate.
from google.colab import files
import csv, io, subprocess, gc, torch
uploaded=files.upload()
assert len(uploaded)==1, 'Upload one inputs-only CSV.'
name, raw=next(iter(uploaded.items()))
rows=list(csv.DictReader(io.StringIO(raw.decode('utf-8-sig'))))
assert rows and set(rows[0])=={'case_id','english','chinese'}
input_path=Path('/content/v11_inputs.csv')
input_path.write_bytes(raw)
# Release interactive model memory before starting the frozen benchmark runner.
del check_notice, checker
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
subprocess.run([sys.executable,str(folder/'run.py'),'--inputs',str(input_path),
    '--output','/content/v11_predictions.json','--cache','/content/v11_cache.json',
    '--require-frozen'],check=True)
files.download('/content/v11_predictions.json')
